# API feasibility rerun - Kaggle version (SYNTHETIC text only)
Run the cells **top to bottom**. Only cell **C** needs your input (today's usage numbers). Keys are typed into hidden boxes.

**Before you start:** Kaggle notebook settings -> **Internet: On**. No GPU needed.

**Built-in safety:** free tier only; a model that needs payment, fails authentication or hits its daily quota is stopped; a 503 is never read as a quota signal; hard budgets cap total requests, total tokens, requests per model and requests per model per lecture; create an empty file named `STOP` in `/kaggle/working/pilot_out_v2` (or press Interrupt) to stop everything; results are resumable (run the run-cell again, nothing finished is repeated). Only synthetic text is ever sent.

**Cells:** A = write code (no network) - B = keys - C = usage + preflight (lists models only, no generation) - D = RUN (real API calls) - E = analyse and zip.

### A - write the project files (no network, no keys)

In [1]:
!mkdir -p pilot_project/src/lecsum_pilot pilot_project/tools

In [2]:
%%writefile pilot_project/src/lecsum_pilot/__init__.py
"""API feasibility pilot (SYNTHETIC text only). Separate from the extraction package `lecsum`."""
__version__ = "0.1.0-pilot"


Writing pilot_project/src/lecsum_pilot/__init__.py


In [3]:
%%writefile pilot_project/src/lecsum_pilot/synth.py
"""Synthetic lecture generator. No real lecture, file or university material is ever read.

Lectures are built from INVENTED entities and numbers, so a model cannot know the facts and
every fact is known by construction (gold facts, planted fabricated claims, planted distortions).
Deterministic: same (subject, target_tokens, seed) -> identical text."""
from __future__ import annotations

import random
import re
from dataclasses import dataclass, field
from typing import Dict, List

SYL = ["ka", "lo", "mi", "ra", "ven", "tor", "sa", "del", "no", "fi", "gar", "pel", "zu", "bri", "han", "quo", "tex", "ur"]

SUBJECTS: Dict[str, Dict[str, List[str]]] = {
    "databases": {
        "fact": ["The {A} index stores {n} keys per page and splits when it reaches {m} entries.",
                 "In the {A} protocol a transaction must acquire {n} locks before the commit phase.",
                 "The {A} join algorithm needs {n} passes over the {B} table when memory holds {m} blocks.",
                 "A {A} log record is {n} bytes long and is flushed every {m} milliseconds."],
        "filler": ["Students often confuse {A} with {B}, but they answer different design questions.",
                   "Consider a small example with {B} to see how {A} behaves under load.",
                   "The lecture returns to {A} later when concurrency is discussed."]},
    "operating_systems": {
        "fact": ["The {A} scheduler uses a time slice of {n} milliseconds and demotes a process after {m} slices.",
                 "A {A} page table has {n} levels and each level indexes {m} entries.",
                 "The {A} allocator rounds requests up to multiples of {n} bytes and keeps {m} free lists.",
                 "Under {A}, a blocked thread is woken after at most {n} timer ticks."],
        "filler": ["Compare {A} with {B} when thinking about fairness between processes.",
                   "A diagram of {A} usually helps, so the instructor draws it on the board.",
                   "Exercises on {B} appear in the second problem set."]},
    "microeconomics": {
        "fact": ["In the {A} model the price elasticity of demand is {n} percent at a price of {m} units.",
                 "The {A} firm earns zero profit when its output reaches {n} units and marginal cost equals {m}.",
                 "A {A} tax of {n} percent shifts the supply curve by {m} units.",
                 "The {A} market clears at {n} units when the subsidy is {m} percent."],
        "filler": ["Do not mix up {A} with {B}; the assumptions about buyers differ.",
                   "The textbook illustrates {A} with a market for {B}.",
                   "Be ready to sketch {A} in the exam."]},
    "organic_chemistry": {
        "fact": ["The {A} reaction proceeds at {n} degrees and gives a yield of {m} percent.",
                 "Compound {A} has {n} stereocentres and a melting point of {m} degrees.",
                 "The {A} catalyst is used at {n} mol percent and is recovered after {m} cycles.",
                 "In the {A} synthesis the intermediate is stable for {n} hours at {m} degrees."],
        "filler": ["The mechanism of {A} resembles {B}, though the intermediates differ.",
                   "Draw {A} carefully; arrows show electron movement.",
                   "Lab session {B} practises this technique."]},
    "signals_systems": {
        "fact": ["The {A} filter has a cutoff of {n} hertz and rolls off at {m} decibels per octave.",
                 "A {A} signal is sampled at {n} kilohertz using {m} bits per sample.",
                 "The {A} transform needs {n} multiplications for a window of {m} samples.",
                 "The {A} system has {n} poles and its settling time is {m} milliseconds."],
        "filler": ["Relate {A} to {B} using the frequency response.",
                   "The animation of {A} on the course page is worth watching.",
                   "Notation for {B} follows the previous lecture."]},
    "machine_learning": {
        "fact": ["The {A} model uses {n} layers with {m} units each.",
                 "Training {A} takes {n} epochs with a learning rate of {m} thousandths.",
                 "The {A} regulariser removes {n} percent of weights below a threshold of {m} hundredths.",
                 "A {A} batch holds {n} examples and the validation set holds {m} examples."],
        "filler": ["Compare {A} with {B} on a small dataset before trusting any claim.",
                   "Overfitting is the topic of the next slide on {B}.",
                   "The notebook for {A} will be shared after class."]},
}


def _name(rng: random.Random) -> str:
    return "".join(rng.choice(SYL) for _ in range(rng.choice((2, 3)))).capitalize()


@dataclass
class Fact:
    fact_id: str
    text: str
    template_index: int
    A: str
    B: str
    n: int
    m: int


@dataclass
class Lecture:
    lecture_id: str
    subject: str
    seed: int
    target_tokens: int
    text: str
    facts: List[Fact] = field(default_factory=list)
    fabricated: List[str] = field(default_factory=list)   # plausible claims NOT supported by the text
    distorted: List[Dict] = field(default_factory=list)   # {fact_id, text}: a fact with a changed number

    @property
    def n_words(self) -> int:
        return len(self.text.split())


def make_lecture(subject: str, target_tokens: int, seed: int, tokens_per_word: float = 1.35,
                 lecture_id: str = None) -> Lecture:
    spec = SUBJECTS[subject]
    rng = random.Random(f"{subject}-{target_tokens}-{seed}")
    target_words = int(target_tokens / tokens_per_word)
    lec = Lecture(lecture_id or f"syn-{subject}-{target_tokens}-{seed}", subject, seed, target_tokens, "")
    paras, words, k = [f"Lecture on {subject.replace('_', ' ')} (synthetic)"], 0, 0
    used = set()
    while words < target_words:
        k += 1
        A, B = _name(rng), _name(rng)
        while A in used or B in used or A == B:
            A, B = _name(rng), _name(rng)
        used.update((A, B))
        n, m = rng.randint(3, 96), rng.randint(5, 990)
        ti = rng.randrange(len(spec["fact"]))
        fact_text = spec["fact"][ti].format(A=A, B=B, n=n, m=m)
        lec.facts.append(Fact(f"f{len(lec.facts) + 1:03d}", fact_text, ti, A, B, n, m))
        fillers = [f.format(A=A, B=B) for f in rng.sample(spec["filler"], k=len(spec["filler"]))]
        para = " ".join([f"Section {k}. " + fact_text] + fillers[:rng.choice((2, 3))])
        paras.append(para)
        words += len(para.split())
    lec.text = "\n\n".join(paras)
    # planted claims (never in the text): fresh names; and distortions of real facts
    frng = random.Random(f"{subject}-{target_tokens}-{seed}-planted")
    for _ in range(3):
        A, B = _name(frng) + "x", _name(frng) + "z"          # suffix guarantees the names are absent
        lec.fabricated.append(spec["fact"][frng.randrange(len(spec["fact"]))].format(
            A=A, B=B, n=frng.randint(3, 96), m=frng.randint(5, 990)))
    for f in frng.sample(lec.facts, k=min(3, len(lec.facts))):
        lec.distorted.append({"fact_id": f.fact_id, "text": spec["fact"][f.template_index].format(
            A=f.A, B=f.B, n=f.n + frng.randint(2, 9), m=f.m + frng.randint(11, 90))})
    return lec


# default pilot set: varied subjects and lengths (target tokens are PLANNING sizes; real tokens are measured)
PILOT_SET = [("databases", 1500), ("operating_systems", 2500), ("microeconomics", 4000),
             ("organic_chemistry", 6000), ("signals_systems", 9000), ("machine_learning", 14000)]


def pilot_lectures(seed: int = 7) -> List[Lecture]:
    return [make_lecture(s, t, seed) for s, t in PILOT_SET]


_SENT = re.compile(r"(?<=[.!?])\s+")


def sentences(text: str) -> List[str]:
    return [s.strip() for s in _SENT.split(text) if s.strip()]


def chunk_text(text: str, chunk_words: int) -> List[str]:
    """Paragraph-respecting chunks of about `chunk_words` words (fixed for every model)."""
    chunks, cur, n = [], [], 0
    for para in text.split("\n\n"):
        w = len(para.split())
        if cur and n + w > chunk_words:
            chunks.append("\n\n".join(cur)); cur, n = [], 0
        cur.append(para); n += w
    if cur:
        chunks.append("\n\n".join(cur))
    return chunks


# ---------------------------------------------------------------------------------------------
# Judge-validation material: planted errors of KNOWN type (all synthetic, deterministic)
# ---------------------------------------------------------------------------------------------
FAITH_CATEGORIES = ["supported_exact", "supported_reworded", "fabricated", "number_distortion",
                    "entity_substitution", "negated_relationship"]
# which verdict(s) count as correct; "flag" = anything other than "supported" is a correct detection
EXPECTED = {"supported_exact": "supported", "supported_reworded": "supported", "fabricated": "unsupported",
            "number_distortion": "contradicted", "entity_substitution": "contradicted",
            "negated_relationship": "contradicted"}
COMPL_CATEGORIES = ["covered", "omitted", "distorted_in_summary"]


def _lower_first(s: str) -> str:
    return s[0].lower() + s[1:] if s else s


def faithfulness_items(lec: Lecture, per_category: int = 10) -> List[Dict]:
    """Claims with known ground truth. None of the erroneous claims occurs in the lecture text."""
    spec = SUBJECTS[lec.subject]
    rng = random.Random(f"{lec.lecture_id}-faith-validation")
    facts = lec.facts
    items: List[Dict] = []

    def add(cat, text):
        if cat.startswith("supported") or text not in lec.text:        # erroneous claims must not be in the source
            items.append({"text": text, "category": cat, "expected": EXPECTED[cat]})

    for f in rng.sample(facts, k=min(per_category, len(facts))):
        add("supported_exact", f.text)
    for f in rng.sample(facts, k=min(per_category // 2, len(facts))):
        add("supported_reworded", "According to the lecture, " + _lower_first(f.text))
    for _ in range(per_category):
        A, B = _name(rng) + "x", _name(rng) + "z"                  # suffix guarantees the names are absent
        add("fabricated", spec["fact"][rng.randrange(len(spec["fact"]))].format(
            A=A, B=B, n=rng.randint(3, 96), m=rng.randint(5, 990)))
    for f in rng.sample(facts, k=min(per_category, len(facts))):
        add("number_distortion", spec["fact"][f.template_index].format(
            A=f.A, B=f.B, n=f.n + rng.randint(2, 9), m=f.m + rng.randint(11, 90)))
    for f in rng.sample(facts, k=min(per_category, len(facts))):    # real entity, but another fact's entity name
        other = rng.choice([g for g in facts if g.A != f.A])
        add("entity_substitution", spec["fact"][f.template_index].format(A=other.A, B=f.B, n=f.n, m=f.m))
    forms = ["It is not true that {}", "Contrary to a common assumption, it is false that {}"]
    for k, f in enumerate(rng.sample(facts, k=min(per_category, len(facts)))):
        add("negated_relationship", forms[k % 2].format(_lower_first(f.text)))
    rng.shuffle(items)
    for i, it in enumerate(items):
        it["i"] = i
    return items


def completeness_case(lec: Lecture, per_category: int = 12) -> Dict:
    """A hand-built summary plus facts whose coverage status is known:
    covered (stated in the summary), omitted (absent), distorted_in_summary (present with changed numbers: NOT covered)."""
    spec = SUBJECTS[lec.subject]
    rng = random.Random(f"{lec.lecture_id}-compl-validation")
    m = max(2, min(per_category, len(lec.facts) // 3))
    pool = rng.sample(lec.facts, k=3 * m)
    covered, omitted, distorted = pool[:m], pool[m:2 * m], pool[2 * m:]
    lines = [f"- {f.text}" for f in covered]
    lines += ["- " + spec["fact"][f.template_index].format(A=f.A, B=f.B, n=f.n + rng.randint(2, 9), m=f.m + rng.randint(11, 90))
              for f in distorted]
    fact_texts = {f.text for f in lec.facts}                  # filler must never contain any real fact (ground truth!)
    filler = [s for s in sentences(lec.text) if "Section" not in s and not s.startswith("Lecture on") and s not in fact_texts]
    lines += [f"- {s}" for s in rng.sample(filler, k=min(8, len(filler)))]
    rng.shuffle(lines)
    facts = ([{"fact_id": f.fact_id, "text": f.text, "category": "covered", "expected": True} for f in covered]
             + [{"fact_id": f.fact_id, "text": f.text, "category": "omitted", "expected": False} for f in omitted]
             + [{"fact_id": f.fact_id, "text": f.text, "category": "distorted_in_summary", "expected": False} for f in distorted])
    rng.shuffle(facts)
    for i, f in enumerate(facts):
        f["i"] = i
    return {"summary": "Summary (hand-built for judge validation)\n" + "\n".join(lines), "facts": facts}


Writing pilot_project/src/lecsum_pilot/synth.py


In [4]:
%%writefile pilot_project/src/lecsum_pilot/prompts.py
"""Fixed prompts shared by every model (the only model-specific differences are API parameters,
which the runner records). Synthetic text only."""
import json
import re

SYSTEM_GEN = ("You are a careful university teaching assistant. Use ONLY the text you are given. "
              "Do not add facts, numbers or names that are not in the text.")
SYSTEM_JUDGE = ("You are a strict fact-checking assistant. Answer with valid JSON only: no prose, "
                "no code fences, no explanations outside the JSON.")


def map_prompt(chunk: str) -> str:
    return ("Summarise the lecture excerpt below as concise bullet points (one fact per bullet). "
            "Keep every number and name exactly as written.\n\nEXCERPT:\n" + chunk)


def reduce_prompt(parts: str, words: int) -> str:
    return (f"Combine the partial summaries below into one structured lecture summary of at most {words} words, "
            "with short headed sections and bullet points. Keep every number and name exactly as given; "
            "do not add anything new.\n\nPARTIAL SUMMARIES:\n" + parts)


def final_prompt(text: str, words: int) -> str:
    return (f"Write a structured summary of the lecture below in at most {words} words, with short headed "
            "sections and bullet points. Keep every number and name exactly as written; do not add anything "
            "new.\n\nLECTURE:\n" + text)


def extract_claims_prompt(summary: str) -> str:
    return ("List the factual claims made in the summary below, one self-contained sentence per claim "
            "(include the names and numbers). Return a JSON array of strings.\n\nSUMMARY:\n" + summary)


def faithfulness_prompt(claims, evidence: str) -> str:
    numbered = "\n".join(f"{i}. {c}" for i, c in claims)
    return ("Check each numbered claim against the SOURCE EXCERPTS only. Verdicts: "
            '"supported" (the excerpts state it), "contradicted" (the excerpts state something different), '
            '"unsupported" (the excerpts do not mention it). Return a JSON array like '
            '[{"i": 0, "verdict": "supported"}], one item per claim.\n\nSOURCE EXCERPTS:\n'
            + evidence + "\n\nCLAIMS:\n" + numbered)


def completeness_prompt(facts, summary: str) -> str:
    numbered = "\n".join(f"{i}. {f}" for i, f in facts)
    return ("Check each numbered fact: is it clearly stated in the SUMMARY (same names and numbers)? "
            'Return a JSON array like [{"i": 0, "covered": true}], one item per fact.\n\nSUMMARY:\n'
            + summary + "\n\nFACTS:\n" + numbered)


def gold_prompt(chunk: str) -> str:
    return ("Extract the key facts of the lecture excerpt below, one self-contained sentence per fact, "
            "keeping names and numbers exactly. Return a JSON array of strings.\n\nEXCERPT:\n" + chunk)


_FENCE = re.compile(r"^```(?:json)?\s*|\s*```$", re.M)
_THINK = re.compile(r"<think>.*?</think>", re.S | re.I)


def parse_json(text: str):
    """Extract the first JSON array/object from a model reply; raises ValueError if none."""
    t = _FENCE.sub("", _THINK.sub("", text or "")).strip()
    for opener, closer in (("[", "]"), ("{", "}")):
        i = t.find(opener)
        while i != -1:
            depth, in_str, esc = 0, False, False
            for j in range(i, len(t)):
                c = t[j]
                if in_str:
                    esc = (c == "\\" and not esc)
                    if c == '"' and not esc:
                        in_str = False
                    continue
                if c == '"':
                    in_str = True
                elif c == opener:
                    depth += 1
                elif c == closer:
                    depth -= 1
                    if depth == 0:
                        try:
                            return json.loads(t[i:j + 1])
                        except ValueError:
                            break
            i = t.find(opener, i + 1)
    raise ValueError("no JSON found")


Writing pilot_project/src/lecsum_pilot/prompts.py


In [5]:
%%writefile pilot_project/src/lecsum_pilot/providers.py
"""Minimal provider adapters (standard library only). The API key is read from the environment or
passed in by the caller, is sent ONLY in a request header, and is never logged or printed."""
from __future__ import annotations

import json
import re
import time
import urllib.error
import urllib.request
from dataclasses import dataclass, field
from typing import Dict, List, Optional

USER_AGENT = "lecsum-feasibility-pilot/0.1"
GROQ_BASE = "https://api.groq.com/openai/v1"
GEMINI_BASE = "https://generativelanguage.googleapis.com/v1beta"
KEEP_HEADERS = re.compile(r"^(x-ratelimit-.*|retry-after|date|x-request-id|content-type)$", re.I)
_SECRET = re.compile(r"(AIza[0-9A-Za-z_\-]{20,}|gsk_[0-9A-Za-z]{20,}|Bearer\s+\S+)")


def redact(text: str, limit: int = 600) -> str:
    return _SECRET.sub("<redacted>", text or "")[:limit]


@dataclass
class LLMResult:
    ok: bool
    text: str = ""
    status: Optional[int] = None
    finish_reason: Optional[str] = None
    input_tokens: Optional[int] = None
    output_tokens: Optional[int] = None          # as reported (may or may not include reasoning)
    reasoning_tokens: Optional[int] = None       # only if the provider exposes it
    total_tokens: Optional[int] = None
    usage_raw: Dict = field(default_factory=dict)
    headers: Dict[str, str] = field(default_factory=dict)
    latency_s: float = 0.0
    error: str = ""
    reasoning_chars: int = 0                     # length of returned reasoning text, if any
    param_fallbacks: List[str] = field(default_factory=list)
    visible_output_tokens: Optional[int] = None  # tokens of returned text, ONLY when the provider lets us separate it
    reasoning_status: str = "unknown"            # see reasoning_status() below; missing data is never read as zero
    unaccounted_tokens: Optional[int] = None     # total - input - output - reported reasoning (None if not computable)


def usage_accounting(input_tokens, output_tokens, reasoning_tokens, total_tokens, output_includes_reasoning: bool):
    """Split a provider's usage numbers into: visible output, reported reasoning, unaccounted.

    output_includes_reasoning: True for APIs whose completion count already contains reasoning (Groq gpt-oss),
    False for APIs that report thinking separately (Gemini thoughtsTokenCount).
    A missing reasoning figure is NEVER treated as zero: status says whether the totals are at least consistent
    with zero, or prove that unreported tokens were counted."""
    vis = None
    if output_includes_reasoning:
        if reasoning_tokens is not None and output_tokens is not None:
            vis = max(0, output_tokens - reasoning_tokens)
    else:
        vis = output_tokens
    known = (input_tokens or 0) + (output_tokens or 0) + ((reasoning_tokens or 0) if not output_includes_reasoning else 0)
    unacc = (total_tokens - known) if total_tokens is not None and input_tokens is not None and output_tokens is not None else None
    if reasoning_tokens is not None:
        status = "reported"
    elif unacc is None:
        status = "unknown"
    elif unacc > 0:
        status = "not_reported_but_totals_show_hidden_tokens"
    else:
        status = "not_reported_totals_consistent_with_zero_not_proof"
    return vis, status, unacc


def _http(url: str, headers: Dict[str, str], body: Optional[dict], timeout: float = 120):
    data = None if body is None else json.dumps(body).encode()
    req = urllib.request.Request(url, data=data, headers={**headers, "User-Agent": USER_AGENT,
                                 "Content-Type": "application/json"}, method="POST" if body is not None else "GET")
    t = time.time()
    try:
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, dict(r.headers), r.read().decode("utf-8", "replace"), time.time() - t
    except urllib.error.HTTPError as e:
        return e.code, dict(e.headers or {}), (e.read() or b"").decode("utf-8", "replace"), time.time() - t
    except Exception as e:                       # network error / timeout
        return None, {}, f"{type(e).__name__}: {e}", time.time() - t


def _filter(h: Dict[str, str]) -> Dict[str, str]:
    return {k.lower(): v for k, v in h.items() if KEEP_HEADERS.match(k)}


class Provider:
    name = "base"

    def __init__(self, label: str, model_id: str):
        self.label, self.model_id = label, model_id

    def call(self, system: str, user: str, max_tokens: int, extra: Optional[dict] = None) -> LLMResult:
        raise NotImplementedError

    def list_models(self) -> List[str]:
        raise NotImplementedError


class GroqProvider(Provider):
    name = "groq"

    def __init__(self, label, model_id, api_key, temperature=0.0):
        super().__init__(label, model_id)
        self._key, self.temperature = api_key, temperature

    def _auth(self):
        return {"Authorization": f"Bearer {self._key}"}

    def list_models(self):
        s, _, body, _ = _http(f"{GROQ_BASE}/models", self._auth(), None)
        if s != 200:
            raise RuntimeError(f"Groq list models failed: HTTP {s} {redact(body, 200)}")
        return sorted(m["id"] for m in json.loads(body).get("data", []))

    def call(self, system, user, max_tokens, extra=None):
        extra = dict(extra or {})                  # e.g. {"reasoning_effort": "low", "include_reasoning": True}
        body = {"model": self.model_id, "temperature": self.temperature, "max_completion_tokens": max_tokens,
                "messages": [{"role": "system", "content": system}, {"role": "user", "content": user}], **extra}
        fallbacks: List[str] = []
        for _ in range(3):
            s, h, text, lat = _http(f"{GROQ_BASE}/chat/completions", self._auth(), body)
            if s == 400 and extra:
                bad = next((k for k in list(extra) if k in text), None)
                if bad:                            # provider rejects an optional parameter: drop it, record it
                    fallbacks.append(f"dropped param {bad}: {redact(text, 160)}")
                    extra.pop(bad); body.pop(bad, None)
                    continue
                if not fallbacks:                  # 400 not naming a parameter: retry once without ALL extras (recorded)
                    fallbacks.append(f"dropped ALL optional params {sorted(extra)}: {redact(text, 160)}")
                    for k in list(extra):
                        body.pop(k, None)
                    extra.clear()
                    continue
            break
        res = LLMResult(ok=False, status=s, headers=_filter(h), latency_s=round(lat, 3), param_fallbacks=fallbacks)
        if s != 200:
            res.error = redact(text)
            return res
        try:
            j = json.loads(text)
            ch = j["choices"][0]
            msg = ch.get("message", {})
            u = j.get("usage", {}) or {}
            det = u.get("completion_tokens_details") or {}
            res.ok, res.text = True, (msg.get("content") or "")
            res.finish_reason = ch.get("finish_reason")
            res.input_tokens, res.output_tokens = u.get("prompt_tokens"), u.get("completion_tokens")
            res.total_tokens = u.get("total_tokens")
            res.reasoning_tokens = det.get("reasoning_tokens")
            res.visible_output_tokens, res.reasoning_status, res.unaccounted_tokens = usage_accounting(
                res.input_tokens, res.output_tokens, res.reasoning_tokens, res.total_tokens, True)
            res.usage_raw = u
            res.reasoning_chars = len(msg.get("reasoning") or msg.get("reasoning_content") or "")
        except Exception as e:
            res.ok, res.error = False, f"parse error: {type(e).__name__}: {e}"
        return res


class GeminiProvider(Provider):
    name = "gemini"

    def __init__(self, label, model_id, api_key, temperature=None):
        super().__init__(label, model_id)
        self._key, self.temperature = api_key, temperature

    def _auth(self):
        return {"x-goog-api-key": self._key}

    def list_models(self):
        out, token = [], None
        for _ in range(10):
            url = f"{GEMINI_BASE}/models?pageSize=200" + (f"&pageToken={token}" if token else "")
            s, _, body, _ = _http(url, self._auth(), None)
            if s != 200:
                raise RuntimeError(f"Gemini list models failed: HTTP {s} {redact(body, 200)}")
            j = json.loads(body)
            out += [m["name"].split("/", 1)[-1] for m in j.get("models", [])
                    if "generateContent" in m.get("supportedGenerationMethods", [])]
            token = j.get("nextPageToken")
            if not token:
                break
        return sorted(out)

    def call(self, system, user, max_tokens, extra=None):
        extra = dict(extra or {})                  # e.g. {"thinkingConfig": {"thinkingLevel": "minimal"}}
        gen = {"maxOutputTokens": max_tokens}
        if self.temperature is not None:
            gen["temperature"] = self.temperature
        gen.update(extra)
        body = {"systemInstruction": {"parts": [{"text": system}]},
                "contents": [{"role": "user", "parts": [{"text": user}]}], "generationConfig": gen}
        fallbacks: List[str] = []
        url = f"{GEMINI_BASE}/models/{self.model_id}:generateContent"
        for _ in range(3):
            s, h, text, lat = _http(url, self._auth(), body)
            if s == 400:
                bad = next((k for k in list(extra) if k.lower() in text.lower()), None)
                if bad:
                    fallbacks.append(f"dropped param {bad}: {redact(text, 160)}")
                    extra.pop(bad); gen.pop(bad, None)
                    continue
                if not fallbacks and extra:        # 400 not naming a parameter: retry once without ALL extras (recorded)
                    fallbacks.append(f"dropped ALL optional params {sorted(extra)}: {redact(text, 160)}")
                    for k in list(extra):
                        gen.pop(k, None)
                    extra.clear()
                    continue
            break
        res = LLMResult(ok=False, status=s, headers=_filter(h), latency_s=round(lat, 3), param_fallbacks=fallbacks)
        if s != 200:
            res.error = redact(text)
            return res
        try:
            j = json.loads(text)
            cand = (j.get("candidates") or [{}])[0]
            parts = (cand.get("content") or {}).get("parts") or []
            u = j.get("usageMetadata", {}) or {}
            res.ok = True
            res.text = "".join(p.get("text", "") for p in parts if not p.get("thought"))
            res.finish_reason = cand.get("finishReason")
            res.input_tokens, res.output_tokens = u.get("promptTokenCount"), u.get("candidatesTokenCount")
            res.reasoning_tokens, res.total_tokens = u.get("thoughtsTokenCount"), u.get("totalTokenCount")
            res.visible_output_tokens, res.reasoning_status, res.unaccounted_tokens = usage_accounting(
                res.input_tokens, res.output_tokens, res.reasoning_tokens, res.total_tokens, False)
            res.usage_raw = u
            if not res.text and j.get("promptFeedback"):
                res.ok, res.error = False, "blocked: " + redact(json.dumps(j["promptFeedback"]), 200)
        except Exception as e:
            res.ok, res.error = False, f"parse error: {type(e).__name__}: {e}"
        return res


def resolve_model_id(wanted: str, available: List[str]) -> Dict:
    """Pick the exact ID for a wanted name from the provider's live model list (never guessed)."""
    if wanted in available:
        return {"wanted": wanted, "resolved": wanted, "alternatives": []}
    bad = ("image", "tts", "live", "audio", "transcribe", "embedding", "robotics", "customtools")
    cands = [a for a in available if a.startswith(wanted) and not any(b in a for b in bad)]
    cands.sort(key=len)
    return {"wanted": wanted, "resolved": cands[0] if cands else None, "alternatives": cands[1:6]}


Writing pilot_project/src/lecsum_pilot/providers.py


In [6]:
%%writefile pilot_project/src/lecsum_pilot/validators.py
"""Pure validators for model replies. Every reply is checked BEFORE it is accepted.

Each validator returns (status, parsed, reason) with status in
  "parsed"     - usable
  "malformed"  - not the requested structure (not JSON, wrong types, unknown labels, ungrounded content)
  "incomplete" - structure fine so far but cut off or missing items (truncated by the output ceiling,
                 missing ids, numbers from the input absent from the output)
so the run log can count the three separately."""
from __future__ import annotations

import re
from typing import Iterable, List, Optional

from .prompts import parse_json

_THINK = re.compile(r"<think>.*?</think>", re.S | re.I)
_NUM = re.compile(r"\d+(?:\.\d+)?")


def truncated(finish: Optional[str]) -> bool:
    return (finish or "").lower() in ("length", "max_tokens")


def _overlap(a: str, b: str) -> float:
    ta, tb = set(re.findall(r"\w+", a.lower())), set(re.findall(r"\w+", b.lower()))
    return len(ta & tb) / (len(ta) or 1)


def v_text(text: str, finish: Optional[str] = None, min_words: int = 5):
    t = _THINK.sub("", text or "").strip()
    if truncated(finish):
        return "incomplete", t, "cut off by the output ceiling (finish_reason=%s)" % finish
    if len(t.split()) < min_words:
        return "malformed", t, "fewer than %d words" % min_words
    return "parsed", t, ""


def _load(text: str, finish: Optional[str]):
    try:
        return parse_json(text), None
    except ValueError:
        return None, ("incomplete", "JSON cut off by the output ceiling (finish_reason=%s)" % finish) if truncated(finish) \
            else ("malformed", "no valid JSON")


def v_string_list(text: str, finish: Optional[str] = None, source_text: Optional[str] = None,
                  min_items: int = 1, number_recall: float = 0.7, min_words: int = 4):
    """List of claim/fact strings. If source_text is given the claims must be grounded in it and must
    reproduce most of its numbers (a quick check that extraction did not stop early)."""
    data, err = _load(text, finish)
    if err:
        return err[0], None, err[1]
    if not isinstance(data, list) or not all(isinstance(x, str) for x in data):
        return "malformed", None, "expected a JSON array of strings"
    items = [x.strip() for x in data if len(x.split()) >= min_words]
    if source_text is not None:
        grounded = [x for x in items if _overlap(x, source_text) >= 0.5]
        if items and len(grounded) < 0.5 * len(items):
            return "malformed", None, "most claims are not grounded in the text"
        items = grounded
    if len(items) < min_items:
        return "incomplete", None, "fewer than %d usable items" % min_items
    if source_text is not None:
        nums = set(_NUM.findall(source_text))
        if nums:
            hit = {n for n in nums if any(n in c for c in items)}
            if len(hit) / len(nums) < number_recall:
                return "incomplete", None, "only %d/%d source numbers appear in the claims" % (len(hit), len(nums))
    return "parsed", items, ""


def v_verdicts(text: str, finish: Optional[str], ids: Iterable[int], allowed=("supported", "contradicted", "unsupported")):
    ids = set(ids)
    data, err = _load(text, finish)
    if err:
        return err[0], None, err[1]
    if not isinstance(data, list):
        return "malformed", None, "expected a JSON array"
    out = {}
    for x in data:
        try:
            i, v = int(x["i"]), str(x["verdict"]).lower().strip()
        except Exception:
            return "malformed", None, "item without i/verdict"
        if v not in allowed:
            return "malformed", None, "unknown verdict %r" % v
        if i in ids and i not in out:
            out[i] = v
    missing = sorted(ids - set(out))
    if missing:
        return "incomplete", out, "no verdict for ids %s" % missing[:8]
    return "parsed", out, ""


def v_covered(text: str, finish: Optional[str], ids: Iterable[int]):
    ids = set(ids)
    data, err = _load(text, finish)
    if err:
        return err[0], None, err[1]
    if not isinstance(data, list):
        return "malformed", None, "expected a JSON array"
    out = {}
    for x in data:
        try:
            i, c = int(x["i"]), x["covered"]
        except Exception:
            return "malformed", None, "item without i/covered"
        if not isinstance(c, bool):
            return "malformed", None, "covered must be true/false"
        if i in ids and i not in out:
            out[i] = c
    missing = sorted(ids - set(out))
    if missing:
        return "incomplete", out, "no answer for ids %s" % missing[:8]
    return "parsed", out, ""


_BULLET = re.compile(r"^[\s\-\*\u2022#>]+")


def split_lines(text: str, max_lines: int) -> List[str]:
    """Split a summary into batches of at most max_lines content lines. Bullet markers are stripped and lines with
    fewer than 4 real words (headings, 'Section 3.') are dropped, so they cannot count as missing numbers."""
    lines = [_BULLET.sub("", l).strip() for l in text.splitlines()]
    lines = [l for l in lines if len(l.split()) >= 4]
    return ["\n".join(lines[i:i + max_lines]) for i in range(0, len(lines), max_lines)]


Writing pilot_project/src/lecsum_pilot/validators.py


In [7]:
%%writefile pilot_project/src/lecsum_pilot/mock.py
"""Fake provider for TESTING the pilot harness. It makes NO network calls and uses no API key.

It can imitate the real failure modes seen in the first pilot: per-minute input/output limits (429 with the
provider-style message), 503 overload, reasoning that eats the output budget, hidden (unreported) reasoning tokens,
truncated JSON, and garbage replies. Time is virtual (FakeClock) so tests run instantly."""
from __future__ import annotations

import json
import re
from typing import Dict, Optional

from .providers import LLMResult, Provider, usage_accounting

_NUM = re.compile(r"\d+")
_CAP = re.compile(r"\b[A-Z][a-z]{3,}\b")


class FakeClock:
    def __init__(self):
        self.t = 1_000_000.0

    def now(self):
        return self.t

    def sleep(self, s):
        self.t += max(0.0, s)


class MockProvider(Provider):
    name = "mock"
    registry: set = set()                       # every mock model id (the list is provider-wide, like a real API)

    def __init__(self, label, model_id, fail_plan: Optional[Dict[int, str]] = None, reasoning_per_call=0,
                 payment_after: Optional[int] = None, garbage_json_calls=(), limits: Optional[Dict[str, int]] = None,
                 report_reasoning=True, output_includes_reasoning=True, hidden_tokens_per_call=0,
                 overload_first_n=0, truncate_json_calls=(), tpm_header=8000, no_compress=False, verbosity=None, interrupt_at=None, overrun=None):
        super().__init__(label, model_id)
        MockProvider.registry.add(model_id)
        self.calls = 0
        self.fail_plan = fail_plan or {}          # {call_number: "429_rate" | "429_daily" | "500" | "503" | "402" | "401"}
        self.reasoning_per_call = reasoning_per_call
        self.report_reasoning = report_reasoning
        self.output_includes_reasoning = output_includes_reasoning
        self.hidden_tokens_per_call = hidden_tokens_per_call     # counted in total_tokens but never reported
        self.payment_after = payment_after
        self.garbage_json_calls = set(garbage_json_calls)
        self.truncate_json_calls = set(truncate_json_calls)
        self.overload_first_n = overload_first_n
        self.limits = limits or {}                # {"rpm","tpm","itpm","otpm"} enforced on a sliding 60 s window
        self.events = []                          # (t, tin, tout) of accepted calls
        self.rate_limited = 0
        self.time_fn = lambda: 0.0                # set by mock_pilot to the pilot's virtual clock
        self.tpm_header = tpm_header
        self.no_compress = no_compress              # imitate a model that ignores the word target (long reduce outputs)
        self.overrun = overrun                      # reduce/final calls write overrun x the requested word target (models that ignore it)
        self.interrupt_at = interrupt_at            # raise KeyboardInterrupt on this call number (the notebook's Interrupt button)
        self.verbosity = verbosity                  # None, or output length = verbosity x input length for every summary call (ignores word targets)
        self.max_seen_tokens = {"in": 0, "out": 0}

    def list_models(self):
        return sorted(MockProvider.registry | {self.model_id + "-other"})

    def _fail(self, kind, msg=None):
        body = {"429_rate": "Rate limit reached for model on tokens per minute (TPM): Limit 8000",
                "429_daily": "Rate limit reached on tokens per day (TPD): Limit 200000, Used 199990",
                "500": "internal error", "503": "This model is currently experiencing high demand.",
                "402": "payment required: billing is not enabled", "401": "invalid API key"}[kind]
        st = int(kind[:3])
        return LLMResult(ok=False, status=st, error=msg or body, headers={"retry-after": "1"} if st == 429 else {})

    def _enforce(self, tin, tout):
        now = self.time_fn()
        live = [e for e in self.events if e[0] > now - 60]
        checks = [("requests per minute", "RPM", len(live) + 1, self.limits.get("rpm")),
                  ("input tokens per minute", "ITPM", sum(e[1] for e in live) + tin, self.limits.get("itpm")),
                  ("output tokens per minute", "OTPM", sum(e[2] for e in live) + tout, self.limits.get("otpm")),
                  ("tokens per minute", "TPM", sum(e[1] + e[2] for e in live) + tin + tout, self.limits.get("tpm"))]
        for name, dim, used_plus_req, lim in checks:
            if lim and used_plus_req > lim:
                self.rate_limited += 1
                r = self._fail("429_rate", f"Rate limit reached for model `{self.model_id}` on {name} ({dim}): "
                                           f"Limit {lim}, Used {used_plus_req - (tin if dim == 'ITPM' else tout if dim == 'OTPM' else 0)}, Requested {tin}")
                r.headers = {"retry-after": "5"}
                return r
        self.events.append((now, tin, tout))
        return None

    def call(self, system, user, max_tokens, extra=None):
        self.calls += 1
        if self.interrupt_at is not None and self.calls == self.interrupt_at:
            raise KeyboardInterrupt()
        if self.payment_after is not None and self.calls > self.payment_after:
            return self._fail("402")
        if self.calls <= self.overload_first_n:
            return self._fail("503")
        if self.calls in self.fail_plan:
            return self._fail(self.fail_plan[self.calls])
        text = self._answer(user)
        if self.calls in self.garbage_json_calls:
            text = "Sure! here you go (not json)"
        tin = max(1, len(system + user) // 4)
        tvis = max(1, len(text) // 4)
        finish = "stop"
        reasoning = self.reasoning_per_call
        if "reasoning_effort" in (extra or {}) and extra["reasoning_effort"] == "low":
            reasoning = reasoning // 4
        if reasoning >= max_tokens:                                  # all of the ceiling spent on reasoning
            text, tvis, finish = "", 0, "length"
            reasoning = max_tokens
        elif tvis + reasoning > max_tokens:
            keep = max(1, (max_tokens - reasoning) * 4)
            text, tvis, finish = text[:keep], max(1, keep // 4), "length"
        if self.calls in self.truncate_json_calls and text:
            text, finish = text[: max(1, len(text) // 2)], "length"
        tout = tvis + (reasoning if self.output_includes_reasoning else 0)
        self.max_seen_tokens["in"], self.max_seen_tokens["out"] = max(self.max_seen_tokens["in"], tin), max(self.max_seen_tokens["out"], tout)
        lim = self._enforce(tin, tout)
        if lim is not None:
            return lim
        reported = reasoning if (self.report_reasoning and reasoning) else None
        total = tin + tout + (0 if self.output_includes_reasoning else reasoning) + self.hidden_tokens_per_call
        vis, status, unacc = usage_accounting(tin, tout, reported, total, self.output_includes_reasoning)
        fb = [f"dropped param {k}" for k in (extra or {}) if k == "unsupported_param"]
        now = self.time_fn()
        rem = max(0, self.tpm_header - sum(e[1] + e[2] for e in self.events if e[0] > now - 60))
        return LLMResult(ok=True, text=text, status=200, finish_reason=finish, input_tokens=tin,
                         output_tokens=tout if self.output_includes_reasoning else tvis, reasoning_tokens=reported,
                         total_tokens=total, usage_raw={"mock": True}, latency_s=0.01, param_fallbacks=fb,
                         visible_output_tokens=vis, reasoning_status=status, unaccounted_tokens=unacc,
                         headers={"x-ratelimit-remaining-tokens": str(rem), "x-ratelimit-remaining-requests": str(1000 - self.calls),
                                  "x-ratelimit-reset-tokens": "1s"})

    def _answer(self, user: str) -> str:
        if "List the factual claims" in user:
            summ = user.split("SUMMARY:\n")[-1].split("\n\nReturn ONLY")[0]
            return json.dumps([s.lstrip("- ").strip() for s in summ.splitlines() if _NUM.search(s)][:24])
        if "Extract the key facts" in user:
            body = user.split("EXCERPT:\n")[-1].split("\n\nReturn ONLY")[0]
            return json.dumps([s for s in re.split(r"(?<=[.!?])\s+", body) if _NUM.search(s)][:80])
        if "lecture excerpt below" in user or "LECTURE:" in user or "PARTIAL SUMMARIES:" in user:
            body = user.split("EXCERPT:\n")[-1].split("LECTURE:\n")[-1].split("PARTIAL SUMMARIES:\n")[-1]
            m = re.search(r"at most (\d+) words", user)
            if self.overrun and m:                              # ignores the requested length: overrun x the target (about 6 characters a word)
                n = int(self.overrun * int(m.group(1)) * 6)
                return (body * (n // max(1, len(body)) + 1))[:n]
            if self.verbosity:                                  # a verbose model: writes about verbosity x the length of what it was given
                n = int(self.verbosity * len(body))
                return (body * (n // max(1, len(body)) + 1))[:n]
            if self.no_compress and "PARTIAL SUMMARIES:" in user:
                return body                                     # echoes everything: the output is as long as the input
            sents = re.split(r"(?<=[.!?])\s+", body)
            return "\n".join("- " + s for s in sents if _NUM.search(s))[:3000]
        if "Check each numbered claim" in user:
            ev = user.split("SOURCE EXCERPTS:\n")[1].split("\n\nCLAIMS:")[0]
            out = []
            for line in user.split("CLAIMS:\n")[-1].splitlines():
                m = re.match(r"(\d+)\.\s+(.*)", line)
                if not m:
                    continue
                claim = m.group(2)
                nums_ok = all(n in ev for n in _NUM.findall(claim))
                names_ok = all(c in ev for c in _CAP.findall(claim)[1:])
                v = "supported" if nums_ok and names_ok else ("contradicted" if names_ok else "unsupported")
                out.append({"i": int(m.group(1)), "verdict": v})
            return json.dumps(out)
        if "Check each numbered fact" in user:
            summ = user.split("SUMMARY:\n")[1].split("\n\nFACTS:")[0]
            out = []
            for line in user.split("FACTS:\n")[-1].splitlines():
                m = re.match(r"(\d+)\.\s+(.*)", line)
                if m:
                    out.append({"i": int(m.group(1)), "covered": all(n in summ for n in _NUM.findall(m.group(2)))})
            return json.dumps(out)
        return "ok"


Writing pilot_project/src/lecsum_pilot/mock.py


In [8]:
%%writefile pilot_project/src/lecsum_pilot/runner.py
"""Pilot runner (rerun version): fixed prompts, per-model accounting, rate-limit scheduling, safe stopping, resume.

Safety rules enforced here
  * input text comes ONLY from the synthetic generator (no file is ever opened for content)
  * a model stops on payment/billing errors, auth errors, or an exhausted DAILY quota (never retried through)
  * a 503 / timeout is NEVER read as a quota signal; it is retried with exponential backoff + jitter, and
    exhausted retries are recorded separately (and, after repeated exhaustion, stop that model for this run)
  * per-model caps keep each model below a fraction of its free daily limit
  * every attempt is logged (requests.jsonl); every accepted output is cached (outputs.jsonl) so a re-run resumes
  * every reply is validated before it is accepted; malformed / incomplete / parsed are counted separately
"""
from __future__ import annotations

import json
import math
import random
import re
import threading
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

from . import prompts as P
from . import validators as V
from .providers import LLMResult, Provider, redact
from .synth import (Lecture, chunk_text, completeness_case, faithfulness_items, sentences)

DAILY_RE = re.compile(r"per\s*day|PerDay|\bTPD\b|\bRPD\b|daily", re.I)
PAYMENT_RE = re.compile(r"billing|payment|paid plan|upgrade|credit card", re.I)
RL_RE = re.compile(r"on\s+(.+?)\s*\((\w+)\):\s*Limit\s+(\d+)(?:,\s*Used\s+(\d+))?(?:,\s*Requested\s+(\d+))?", re.I)


@dataclass
class ModelSpec:
    label: str
    provider: str                       # groq | gemini | mock
    wanted_id: str
    family: str                         # openai | qwen | google (generator/judge families are kept distinct)
    rpm: int = 10
    max_tokens_per_run: Optional[int] = None     # self-imposed cap (tokens) for this run
    max_requests_per_run: Optional[int] = None   # per launch (derived from today's dashboard usage)
    budget_requests: Optional[int] = None        # RERUN budget: HTTP attempts for this model, cumulative over every resume in out_dir
    extra: Dict = field(default_factory=dict)    # model-specific API parameters (recorded in the log)
    # provider limits used by the scheduler (None = not applicable / not shown)
    tpm: Optional[int] = None                    # total tokens per minute (input + output)
    itpm: Optional[int] = None                   # input tokens per minute (Qwen on Groq)
    otpm: Optional[int] = None                   # output tokens per minute (Qwen on Groq)
    rate_safety: float = 0.85                    # use at most this share of every per-minute limit
    # per-model protocol settings (recorded; a deviation from the shared default is a documented constraint)
    chunk_words: Optional[int] = None
    map_max_tokens: Optional[int] = None
    reduce_max_tokens: Optional[int] = None
    judge_max_tokens: Optional[int] = None
    summary_word_cap: Optional[int] = None       # e.g. Qwen: summary must fit in a 900-token output ceiling
    max_attempts_overload: int = 6               # 503 / timeout attempts before retries count as exhausted
    overload_circuit: int = 3                    # consecutive exhausted requests that stop the model for this run


@dataclass
class PilotConfig:
    out_dir: str = "pilot_out_v2"
    chunk_words: int = 2200                      # about 3300 tokens (measured 1.45-1.57 tokens per word)
    map_max_tokens: int = 4000
    reduce_max_tokens: int = 4000
    judge_max_tokens: int = 3000
    reduce_input_tokens: int = 3500              # tree-reduce: no reduce call receives more than this (all models)
    claim_batch_lines: int = 10                  # summary lines per claim-extraction call
    claim_max_tokens: int = 1500
    claim_batch: int = 8                         # claims per faithfulness call
    evidence_per_claim: int = 3
    gold_facts_per_check: int = 15
    max_claims: int = 40
    max_attempts_rate: int = 5                   # attempts spent on rate-limit (429, not daily) answers
    max_wait_s: float = 90.0                     # cap on one rate-limit wait
    backoff_base_s: float = 15.0                 # 503 / timeout: base * 2^(n-1), capped, with jitter
    backoff_cap_s: float = 240.0
    max_reduce_levels: int = 6                   # hard safety limit on reduce levels (never reached when packing works)
    max_split_depth: int = 2                     # a truncated map call is split in halves this many times at most
    judged_lecture_idx: tuple = (1, 3)           # operational judging of generated summaries
    gold_lecture_idx: tuple = (0, 2)
    validation_lecture_idx: tuple = (0, 1, 2)    # judge validation (planted errors) lectures
    validation_per_category: int = 10
    seed: int = 20261103
    # ---- safety budgets (a real run refuses to start unless these are set; see Pilot.check_budgets)
    max_total_requests: Optional[int] = None     # HTTP attempts, all models, cumulative over every resume in out_dir
    max_total_tokens: Optional[int] = None       # input+output tokens, all models, cumulative (worst case reserved before each call)
    per_lecture_requests: Dict[str, int] = field(default_factory=dict)   # task family -> max HTTP attempts per (model, lecture)
    lecture_trips_before_model_stop: Optional[int] = None    # emergency stop: a model that trips this many lecture budgets is stopped
    stop_file: str = "STOP"                      # create this file inside out_dir to stop every model before its next request
    allow_unbudgeted: bool = False               # unit tests of unrelated behaviour only; the notebooks never set it
    judges_for: Dict[str, List[str]] = field(default_factory=dict)   # generator provider -> judge labels (first = extractor)


FAMILIES = ('gen', 'gold', 'claims_judge', 'jv', 'probe')


class BudgetError(ValueError):
    pass


def task_family(stage: str) -> str:
    """Budget family of a request stage. Per-lecture budgets are kept separately for each family so that, for example,
    judge validation on a lecture cannot starve that lecture's generation."""
    if stage.startswith('gen_'):
        return 'gen'
    if stage == 'gold_generation':
        return 'gold'
    if stage.startswith('jv_'):
        return 'jv'
    if stage.startswith('probe'):
        return 'probe'
    return 'claims_judge'


def _now():
    return datetime.now(timezone.utc).isoformat(timespec="seconds")


def _tokens_overlap(a: str, b: str) -> float:
    ta, tb = set(re.findall(r"\w+", a.lower())), set(re.findall(r"\w+", b.lower()))
    return len(ta & tb) / (len(ta) or 1)


# ------------------------------------------------------------------------------------ classification
def parse_rate_limit(txt: str) -> Dict:
    """Structured view of a 'rate limit reached' message: dimension (TPM/ITPM/OTPM/RPM/TPD/RPD), limit, used, requested."""
    m = RL_RE.search(txt or "")
    if m:
        d = {"rl_dimension": m.group(2).upper(), "rl_limit": int(m.group(3)),
             "rl_used": int(m.group(4)) if m.group(4) else None, "rl_requested": int(m.group(5)) if m.group(5) else None}
        t = re.search(r"try again in\s+(?:(\d+)h)?(?:(\d+)m)?(?:([\d.]+)s)?", txt or "")
        if t and any(t.groups()):
            d["rl_retry_in_s"] = round(int(t.group(1) or 0) * 3600 + int(t.group(2) or 0) * 60 + float(t.group(3) or 0), 1)
        return d
    t = txt or ""
    if "PerDay" in t:
        return {"rl_dimension": "RPD"}
    if "InputTokens" in t and "PerMinute" in t:
        return {"rl_dimension": "ITPM"}
    if "PerMinute" in t:
        return {"rl_dimension": "RPM"}
    return {}


def classify_detail(res: LLMResult) -> Tuple[str, str]:
    """(kind, subtype). kind: ok | retry | daily_quota | payment | auth | fatal.
    subtype separates rate limits (rate_limit_tpm/itpm/otpm/rpm) from overload (overloaded_503, server_error_5xx,
    timeout_network). A 5xx or a timeout is never a quota signal."""
    if res.ok:
        return "ok", ""
    s, txt = res.status, res.error or ""
    if s is None:
        return "retry", "timeout_network"
    if s == 503:
        return "retry", "overloaded_503"
    if s >= 500:
        return "retry", "server_error_5xx"
    if s == 401:
        return "auth", "auth"
    if s == 402 or (s in (400, 403) and PAYMENT_RE.search(txt)):
        return "payment", "payment"
    if s == 429:
        dim = parse_rate_limit(txt).get("rl_dimension", "")
        if dim in ("TPD", "RPD") or DAILY_RE.search(txt):
            return "daily_quota", "daily_quota"
        return "retry", "rate_limit_" + (dim.lower() if dim else "unspecified")
    if s == 413:
        dim = parse_rate_limit(txt).get("rl_dimension", "")
        return "fatal", "request_too_large_" + (dim.lower() if dim else "unspecified")
    if s == 403:
        return "auth", "auth"
    return "fatal", "http_%s" % s


def classify(res: LLMResult) -> str:
    return classify_detail(res)[0]


def _parse_wait(h: Dict[str, str]) -> Optional[float]:
    v = h.get("retry-after") or h.get("x-ratelimit-reset-tokens")
    if not v:
        return None
    m = re.fullmatch(r"(?:(\d+)m)?(?:([\d.]+)(ms|s))?", v.strip())
    if m and (m.group(1) or m.group(2)):
        return int(m.group(1) or 0) * 60 + (float(m.group(2)) / (1000 if m.group(3) == "ms" else 1) if m.group(2) else 0)
    try:
        return float(v)
    except ValueError:
        return None


# ------------------------------------------------------------------------------------ rate-limit scheduler
class Window:
    """Sliding 60-second window of (time, requests, input tokens, output tokens) for ONE model."""

    def __init__(self):
        self.ev: List[list] = []

    def add(self, t, tin, tout):
        e = [t, 1, tin, tout]
        self.ev.append(e)
        if len(self.ev) > 400:
            self.ev = self.ev[-200:]
        return e

    def wait_needed(self, now, tin, tout, rpm, tpm, itpm, otpm) -> Optional[float]:
        """Seconds to wait so that one more request of (tin, tout) fits every limit; None if it can never fit."""
        def fits(live):
            r, i, o = len(live) + 1, sum(e[2] for e in live) + tin, sum(e[3] for e in live) + tout
            return ((rpm is None or r <= rpm) and (itpm is None or i <= itpm) and (otpm is None or o <= otpm)
                    and (tpm is None or i + o <= tpm))
        if not fits([]):
            return None
        for d in sorted({0.0} | {max(0.0, e[0] + 60.0 - now) for e in self.ev}):
            if fits([e for e in self.ev if e[0] > now + d - 60.0]):
                return d
        return 60.0


class Pilot:
    def __init__(self, cfg: PilotConfig, specs: List[ModelSpec], providers: Dict[str, Provider],
                 sleep: Callable[[float], None] = time.sleep, clock: Callable[[], float] = time.time):
        self.cfg, self.specs = cfg, {s.label: s for s in specs}
        self.providers, self._sleep_fn, self._clock = providers, sleep, clock
        self._offset = 0.0                               # virtual time added when a (fake) sleep returns too early
        self.lock = threading.RLock()
        self.out = Path(cfg.out_dir)
        self.out.mkdir(parents=True, exist_ok=True)
        self.req_path, self.out_path = self.out / "requests.jsonl", self.out / "outputs.jsonl"
        self.cache: Dict[str, dict] = {}
        if self.out_path.exists():
            for line in self.out_path.read_text(encoding="utf-8").splitlines():
                if line.strip():
                    r = json.loads(line)
                    self.cache[r["key"]] = r
        self.stopped: Dict[str, str] = {}                # label -> reason (this run)
        self.stopped_kind: Dict[str, str] = {}           # label -> daily_quota | payment | auth | overload_retries_exhausted | run_cap | model_unavailable
        self.tokens_run = {l: 0 for l in self.specs}
        self.requests_run = {l: 0 for l in self.specs}
        self.windows = {l: Window() for l in self.specs}
        self.cpt_sum = {l: [0.0, 0.0] for l in self.specs}          # learned characters per token [chars, tokens]
        self.out_seen: Dict[Tuple[str, str], List[int]] = defaultdict(list)   # (label, stage) -> observed output tokens
        self.exhausted = {l: defaultdict(int) for l in self.specs}   # retries exhausted, by kind (NOT quota exhaustion)
        self.consec_exhausted = {l: 0 for l in self.specs}
        self.errors: List[str] = []
        self.rows_mem: List[dict] = []                   # every logged attempt of this run (also in requests.jsonl)
        self.last_failure: Dict[str, str] = {}
        self.rng = random.Random(cfg.seed)
        self.seq = 0
        self.started = _now()
        # ---- budget state. Counters are rebuilt from requests.jsonl, so a resume can never spend a budget twice.
        self.total_requests = 0
        self.total_tokens = 0
        self.inflight_tokens = 0
        self.cum_req = {l: 0 for l in self.specs}
        self.lec_req: Dict[Tuple[str, str, str], int] = defaultdict(int)       # (model, lecture, family) -> HTTP attempts
        self.lecture_trips: Dict[str, set] = defaultdict(set)                 # model -> lectures whose budget was exhausted
        self.refusals = 0
        self.emergency: Optional[str] = None
        self._levels: Dict[Tuple[str, str], int] = {}
        if self.req_path.exists():
            for line in self.req_path.read_text(encoding="utf-8").splitlines():
                if not line.strip():
                    continue
                r = json.loads(line)
                if r.get("outcome") in ("local_budget_exceeded", "budget_refused"):
                    continue                                                    # these never reached the provider
                self.total_requests += 1
                self.total_tokens += r.get("total_tokens") or ((r.get("input_tokens") or 0) + (r.get("output_tokens") or 0)) \
                    or (r.get("est_input_tokens") or 0)
                if r.get("model") in self.cum_req:
                    self.cum_req[r["model"]] += 1
                self.lec_req[(r.get("model"), r.get("lecture_id"), task_family(r.get("stage", "")))] += 1

    # ------------------------------------------------------------------ time (real, or virtual in tests)
    def now(self) -> float:
        return self._clock() + self._offset

    def sleep(self, s: float):
        if s <= 0:
            return
        t0 = self._clock()
        self._sleep_fn(s)
        e = self._clock() - t0
        if e < s:                                        # a fake sleep: advance virtual time instead
            with self.lock:
                self._offset += s - e

    # ------------------------------------------------------------------ logging
    def _log(self, row: dict):
        with self.lock:
            self.rows_mem.append(row)
            with self.req_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")

    def _store(self, key, **fields):
        row = {"key": key, "ts": _now(), **fields}
        with self.lock:
            self.cache[key] = row
            with self.out_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
        return row

    def _stop(self, label: str, kind: str, reason: str):
        with self.lock:
            self.stopped.setdefault(label, reason)
            self.stopped_kind.setdefault(label, kind)

    # ------------------------------------------------------------------ token estimation / scheduling
    def est_tokens(self, label: str, chars: int) -> int:
        c, t = self.cpt_sum[label]
        cpt = max(2.5, (c / t) * 0.92) if t >= 500 else 3.3          # conservative until learned
        return int(math.ceil(chars / cpt))

    def _expected_out(self, label, stage, max_tokens) -> int:
        seen = self.out_seen[(label, stage)]
        e = int(1.3 * sum(seen[-4:]) / len(seen[-4:])) + 20 if len(seen) >= 2 else int(0.6 * max_tokens)
        return max(16, min(max_tokens, e))

    def _budget(self, spec: ModelSpec, est_in: int, max_tokens: int) -> Tuple[int, bool, Optional[str]]:
        """Clamp the output ceiling so that the request can fit the provider's limits. Returns (max_tokens, clamped, refusal)."""
        mt, clamped = max_tokens, False
        sf = spec.rate_safety
        if spec.itpm and est_in > spec.itpm * sf:
            return mt, False, f"estimated input {est_in} tokens exceeds {sf:.0%} of ITPM {spec.itpm}"
        if spec.otpm and mt > spec.otpm * 0.95:
            mt, clamped = int(spec.otpm * 0.95), True
        if spec.tpm and est_in + mt > spec.tpm * 0.97:
            mt, clamped = int(spec.tpm * 0.97) - est_in, True
        if mt < 200:
            return mt, clamped, f"no room for an answer: input ~{est_in} tokens vs a per-minute limit"
        return mt, clamped, None

    def _schedule(self, label, spec, est_in, reserve_out) -> Optional[float]:
        lim = lambda v: max(1, int(v * spec.rate_safety)) if v else None
        w = self.windows[label].wait_needed(self.now(), est_in, reserve_out, lim(spec.rpm), lim(spec.tpm), lim(spec.itpm), lim(spec.otpm))
        return w

    # ------------------------------------------------------------------ safety budgets
    def check_budgets(self):
        """Raise BudgetError unless every budget is set. Called by run() and by real_pilot()."""
        if self.cfg.allow_unbudgeted:
            return
        miss = []
        c = self.cfg
        for name in ("max_total_requests", "max_total_tokens", "lecture_trips_before_model_stop"):
            v = getattr(c, name)
            if not isinstance(v, int) or isinstance(v, bool) or v <= 0:
                miss.append(name)
        for fam in FAMILIES:
            v = c.per_lecture_requests.get(fam)
            if not isinstance(v, int) or isinstance(v, bool) or v <= 0:
                miss.append(f"per_lecture_requests[{fam}]")
        for l, sp in self.specs.items():
            if not isinstance(sp.budget_requests, int) or isinstance(sp.budget_requests, bool) or sp.budget_requests <= 0:
                miss.append(f"per_model_requests[{l}]")
        if miss:
            raise BudgetError("Safety budgets are missing or invalid: " + ", ".join(miss) + ". Set every budget explicitly "
                              "(positive whole numbers). Nothing was sent.")

    def emergency_stop(self, reason: str):
        """Stop every model before its next request. Also triggered by creating <out_dir>/STOP."""
        with self.lock:
            self.emergency = self.emergency or reason
            for l in self.specs:
                self._stop(l, "emergency_stop", reason)

    def _gate(self, label: str, lecture_id: str, stage: str, est_in: int, mt: int) -> Optional[Tuple[str, str]]:
        """Decide, under the lock and BEFORE a request is sent, whether it may be sent. Returns (kind, reason) to refuse,
        or None after reserving the attempt. This is the only place a request is released to the provider."""
        if self.cfg.allow_unbudgeted:
            with self.lock:
                self.total_requests += 1
                self.cum_req[label] += 1
                self.lec_req[(label, lecture_id, task_family(stage))] += 1
            return None
        if (self.out / self.cfg.stop_file).exists():
            self.emergency_stop(f"stop file {self.cfg.stop_file} found in {self.out}")
        with self.lock:
            if self.emergency:
                return "emergency_stop", self.emergency
            if label in self.stopped:
                return "stopped", self.stopped[label]
            fam = task_family(stage)
            spec = self.specs[label]
            cap_l = self.cfg.per_lecture_requests[fam]
            if self.lec_req[(label, lecture_id, fam)] >= cap_l:
                self.lecture_trips[label].add(lecture_id)
                if len(self.lecture_trips[label]) >= self.cfg.lecture_trips_before_model_stop:
                    self._stop(label, "emergency_stop_lecture_budget",
                               f"{len(self.lecture_trips[label])} lectures exhausted their per-lecture request budget "
                               f"({sorted(self.lecture_trips[label])}); the model is stopped as a safety measure")
                return "lecture_budget", f"per-lecture request budget {cap_l} for ({label}, {lecture_id}, {fam}) is used up"
            if self.cum_req[label] >= spec.budget_requests:
                self._stop(label, "model_request_budget", f"per-model request budget {spec.budget_requests} used up")
                return "model_request_budget", f"per-model request budget {spec.budget_requests} used up for {label}"
            if self.total_requests >= self.cfg.max_total_requests:
                self.emergency_stop(f"total request budget {self.cfg.max_total_requests} for the whole rerun is used up")
                return "total_request_budget", self.emergency
            worst = est_in + mt
            if self.total_tokens + self.inflight_tokens + worst > self.cfg.max_total_tokens:
                self.emergency_stop(f"total token budget {self.cfg.max_total_tokens} for the whole rerun would be exceeded "
                                    f"(spent {self.total_tokens}, in flight {self.inflight_tokens}, this request up to {worst})")
                return "total_token_budget", self.emergency
            self.total_requests += 1
            self.cum_req[label] += 1
            self.lec_req[(label, lecture_id, fam)] += 1
            self.inflight_tokens += worst
        return None

    def _settle(self, est_in: int, mt: int, used: int):
        with self.lock:
            self.inflight_tokens = max(0, self.inflight_tokens - (est_in + mt))
            self.total_tokens += used

    def budget_state(self) -> dict:
        with self.lock:
            return {"total_requests": self.total_requests, "max_total_requests": self.cfg.max_total_requests,
                    "total_tokens": self.total_tokens, "max_total_tokens": self.cfg.max_total_tokens,
                    "requests_by_model": dict(self.cum_req), "budget_by_model": {l: s.budget_requests for l, s in self.specs.items()},
                    "max_per_lecture_by_family": dict(self.cfg.per_lecture_requests),
                    "max_attempts_used_per_lecture": {f"{k[0]}|{k[1]}|{k[2]}": v for k, v in sorted(self.lec_req.items(), key=lambda kv: -kv[1])[:10]},
                    "lecture_budget_trips": {l: sorted(v) for l, v in self.lecture_trips.items() if v},
                    "refused_before_sending": self.refusals, "emergency": self.emergency}

    # ------------------------------------------------------------------ core request
    def request(self, stage: str, label: str, lecture_id: str, part: str, system: str, user: str,
                max_tokens: int, variant: str = "", extra: Optional[dict] = None,
                validate: Optional[Callable[[str, Optional[str]], Tuple[str, object, str]]] = None,
                retry_hint: str = "") -> Optional[dict]:
        """One logical request (several attempts). Returns the stored row, or None.
        After a None return, self.last_failure[(label)] says why: malformed|incomplete|truncated|empty_output|exhausted|stopped|..."""
        key = f"{stage}|{label}|{lecture_id}|{part}|{variant}"
        with self.lock:
            if key in self.cache:
                return self.cache[key]
        spec, prov = self.specs[label], self.providers[label]
        with self.lock:
            if label in self.stopped:
                self.last_failure[label] = "stopped"
                return None
            if spec.max_tokens_per_run is not None and self.tokens_run[label] >= spec.max_tokens_per_run:
                self._stop(label, "run_cap", "run token cap reached (self-imposed safety cap)")
                return None
            if spec.max_requests_per_run is not None and self.requests_run[label] >= spec.max_requests_per_run:
                self._stop(label, "run_cap", "run request cap reached (self-imposed safety cap)")
                return None
        merged = {**spec.extra, **(extra or {})}
        attempt_user, cur_max = user, max_tokens
        n_rate = n_over = n_invalid = 0
        n_empty = 0
        attempt = 0
        while True:
            attempt += 1
            est_in = self.est_tokens(label, len(system) + len(attempt_user))
            mt, clamped, refusal = self._budget(spec, est_in, cur_max)
            if refusal:                                              # never send a request that cannot fit the limits
                self._log({"seq": self._next_seq(), "ts": _now(), "stage": stage, "model": label, "provider": spec.provider,
                           "lecture_id": lecture_id, "part": part, "variant": variant, "attempt": attempt, "status": None,
                           "ok": False, "outcome": "local_budget_exceeded", "subtype": refusal, "est_input_tokens": est_in,
                           "max_tokens": cur_max, "usage_known": False, "headers": {}, "error": refusal})
                self.last_failure[label] = "local_budget_exceeded"
                return None
            reserve = self._expected_out(label, stage, mt)
            wait = self._schedule(label, spec, est_in, reserve)
            if wait is None:                                         # cannot ever fit: lower the reservation to the clamp
                wait = self._schedule(label, spec, est_in, 0) or 0.0
            self.sleep(wait)
            refused = self._gate(label, lecture_id, stage, est_in, mt)
            if refused:                                              # a budget said no: NOTHING is sent
                with self.lock:
                    self.refusals += 1
                self._log({"seq": self._next_seq(), "ts": _now(), "stage": stage, "model": label, "provider": spec.provider,
                           "lecture_id": lecture_id, "part": part, "variant": variant, "attempt": attempt, "status": None,
                           "ok": False, "outcome": "budget_refused", "subtype": refused[0], "est_input_tokens": est_in,
                           "max_tokens": mt, "usage_known": False, "headers": {}, "error": refused[1]})
                self.last_failure[label] = refused[0]
                return None
            ev = self.windows[label].add(self.now(), est_in, reserve)
            try:
                res = prov.call(system, attempt_user, mt, merged)
            except BaseException:
                self._settle(est_in, mt, est_in)                     # an interrupted call is charged its input, never refunded
                raise
            kind, subtype = classify_detail(res)
            with self.lock:
                self.requests_run[label] += 1
                used = res.total_tokens if res.total_tokens is not None else (res.input_tokens or 0) + (res.output_tokens or 0)
                self.tokens_run[label] += used or 0
            self._settle(est_in, mt, used or (est_in + (res.output_tokens or 0) if res.ok else est_in))
            usage_known = res.input_tokens is not None or res.output_tokens is not None
            ev[2], ev[3] = (res.input_tokens if res.input_tokens is not None else est_in), (res.output_tokens if res.output_tokens is not None else (0 if not res.ok else reserve))
            if res.input_tokens and len(system) + len(attempt_user) > 1500:
                with self.lock:
                    self.cpt_sum[label][0] += len(system) + len(attempt_user)
                    self.cpt_sum[label][1] += res.input_tokens
            if res.ok and res.output_tokens:
                self.out_seen[(label, stage)].append(res.output_tokens)
            empty = kind == "ok" and not (res.text or "").strip()
            vstatus, parsed, vreason = None, None, ""
            if kind == "ok" and not empty:
                if validate is None:
                    vstatus, parsed, vreason = V.v_text(res.text, res.finish_reason)
                else:
                    vstatus, parsed, vreason = validate(res.text, res.finish_reason)
            outcome = ("empty_output" if empty else kind if kind != "ok" else ("ok" if vstatus == "parsed" else vstatus))
            row = {"seq": self._next_seq(), "ts": _now(), "stage": stage, "model": label, "provider": spec.provider,
                   "model_id": getattr(prov, "model_id", ""), "lecture_id": lecture_id, "part": part, "variant": variant,
                   "attempt": attempt, "status": res.status, "ok": kind == "ok", "outcome": outcome, "subtype": subtype,
                   "validation": vstatus, "validation_reason": vreason, "finish_reason": res.finish_reason,
                   "input_tokens": res.input_tokens, "output_tokens": res.output_tokens,
                   "visible_output_tokens": res.visible_output_tokens, "reasoning_tokens": res.reasoning_tokens,
                   "reasoning_status": res.reasoning_status if kind == "ok" else None,
                   "unaccounted_tokens": res.unaccounted_tokens, "total_tokens": res.total_tokens, "usage_known": usage_known,
                   "usage_raw": res.usage_raw, "reasoning_chars": res.reasoning_chars, "latency_s": res.latency_s,
                   "prompt_chars": len(system) + len(attempt_user), "response_chars": len(res.text or ""),
                   "est_input_tokens": est_in, "reserved_output_tokens": reserve, "sched_wait_s": round(wait, 2),
                   "max_tokens_requested": cur_max, "max_tokens": mt, "clamped": clamped, "params": merged,
                   "param_fallbacks": res.param_fallbacks, "headers": res.headers, "error": redact(res.error, 400)}
            if subtype.startswith("rate_limit_") or subtype.startswith("request_too_large") or kind == "daily_quota":
                row.update(parse_rate_limit(res.error))
            # ---- decide what happens next
            if kind == "ok" and vstatus == "parsed":
                row["accepted"] = True
                self._log(row)
                with self.lock:
                    self.consec_exhausted[label] = 0
                    self.last_failure.pop(label, None)
                return self._store(key, stage=stage, model=label, lecture_id=lecture_id, part=part, variant=variant,
                                   text=res.text, parsed=parsed, finish_reason=res.finish_reason, input_tokens=res.input_tokens,
                                   output_tokens=res.output_tokens, reasoning_tokens=res.reasoning_tokens,
                                   visible_output_tokens=res.visible_output_tokens, total_tokens=res.total_tokens)
            if kind == "ok":
                with self.lock:
                    self.consec_exhausted[label] = 0
                trunc = V.truncated(res.finish_reason)
                if empty and n_empty < 1 and mt < self._hard_ceiling(spec, est_in):
                    n_empty += 1                                      # reasoning probably used the budget: ONE retry, 1.5x ceiling
                    cur_max = int(cur_max * 1.5)
                    self._log(row)
                    continue
                if (not empty) and (not trunc) and retry_hint and n_invalid < 1:     # malformed / missing items: ask once more
                    n_invalid += 1
                    attempt_user = user + retry_hint
                    self._log(row)
                    continue
                self._log(row)
                self.last_failure[label] = "truncated" if trunc else ("empty_output" if empty else vstatus)
                return None
            if kind in ("daily_quota", "payment", "auth"):
                self._log(row)
                self._stop(label, kind, {"daily_quota": "free DAILY quota exhausted (429)", "payment": "payment/billing required",
                                         "auth": "authentication/permission error"}[kind] + ": " + redact(res.error, 200))
                self.last_failure[label] = kind
                return None
            if kind == "fatal":
                self._log(row)
                self.last_failure[label] = subtype
                return None
            # ---- retryable: rate limit (follow the server's wait) or overload / network (exponential backoff + jitter)
            if subtype.startswith("rate_limit_"):
                n_rate += 1
                if n_rate >= self.cfg.max_attempts_rate:
                    row["exhausted"] = "rate_limit"
                    self._log(row)
                    self._note_exhausted(label, "rate_limit")
                    self.last_failure[label] = "exhausted"
                    return None
                w = min(max(_parse_wait(res.headers) or 15.0 * n_rate, 3.0), self.cfg.max_wait_s)
                w *= self.rng.uniform(1.0, 1.15)
            else:
                n_over += 1
                if n_over >= spec.max_attempts_overload:
                    row["exhausted"] = subtype
                    self._log(row)
                    self._note_exhausted(label, subtype)
                    self.last_failure[label] = "exhausted"
                    return None
                w = min(self.cfg.backoff_cap_s, self.cfg.backoff_base_s * 2 ** (n_over - 1)) * self.rng.uniform(0.5, 1.0)
            row["backoff_s"] = round(w, 1)
            self._log(row)
            self.sleep(w)

    def _hard_ceiling(self, spec, est_in) -> int:
        return int(spec.tpm * 0.97) - est_in if spec.tpm else 10 ** 9

    def _next_seq(self) -> int:
        with self.lock:
            self.seq += 1
            return self.seq

    def _note_exhausted(self, label, kind):
        """Retries exhausted. This is NOT quota exhaustion; the model is only stopped after repeated exhaustion."""
        with self.lock:
            self.exhausted[label][kind] += 1
            self.consec_exhausted[label] += 1
            if self.consec_exhausted[label] >= self.specs[label].overload_circuit:
                self._stop(label, "overload_retries_exhausted",
                           f"{self.consec_exhausted[label]} consecutive requests exhausted their retries ({kind}). "
                           "This is provider overload/instability, NOT proof that a quota is used up.")

    # ------------------------------------------------------------------ preflight / probes
    def preflight(self) -> dict:
        """List models (no generation) and resolve exact IDs. Writes resolved_models.json."""
        from .providers import resolve_model_id
        info, seen = {}, {}
        for label, spec in self.specs.items():
            prov = self.providers[label]
            key = (spec.provider, type(prov).__name__)
            if key not in seen:
                try:
                    seen[key] = prov.list_models()
                except Exception as e:
                    seen[key] = None
                    info.setdefault("_errors", {})[spec.provider] = redact(str(e), 200)
            avail = seen[key]
            info[label] = resolve_model_id(spec.wanted_id, avail) if avail is not None else {"wanted": spec.wanted_id, "resolved": None}
        (self.out / "resolved_models.json").write_text(json.dumps(info, indent=1), encoding="utf-8")
        return info

    def reasoning_probe(self, lecture: Lecture, variants: Dict[str, Dict[str, dict]]):
        """Same prompt, different reasoning settings (small, one call per variant). Shows how much of the output
        budget a setting consumes and whether the provider reports it."""
        for label, vs in variants.items():
            spec = self.specs[label]
            chunk = chunk_text(lecture.text, spec.chunk_words or self.cfg.chunk_words)[0]
            for vname, extra in vs.items():
                self.request("probe_reasoning", label, lecture.lecture_id, "map0", P.SYSTEM_GEN, P.map_prompt(chunk),
                             spec.map_max_tokens or self.cfg.map_max_tokens, variant=vname, extra=extra)

    # ------------------------------------------------------------------ generation
    def summary_words(self, lec: Lecture, label: Optional[str] = None) -> int:
        w = int(min(700, max(150, 0.06 * lec.n_words)))
        cap = self.specs[label].summary_word_cap if label else None
        return min(w, cap) if cap else w

    def _cfg(self, label, name):
        return getattr(self.specs[label], name) or getattr(self.cfg, name)

    def _map(self, label, lec, text, part, depth=0) -> Optional[List[str]]:
        """Map one chunk. If the answer is cut off by the output ceiling the chunk is split in halves (recorded)."""
        r = self.request("gen_map", label, lec.lecture_id, part, P.SYSTEM_GEN, P.map_prompt(text), self._cfg(label, "map_max_tokens"),
                         validate=lambda t, f: V.v_text(t, f))
        if r:
            return [r["text"].strip()]
        if self.last_failure.get(label) in ("truncated", "empty_output", "incomplete") and depth < self.cfg.max_split_depth:
            paras = text.split("\n\n")
            if len(paras) >= 2:
                h = len(paras) // 2
                out = []
                for k, piece in enumerate(("\n\n".join(paras[:h]), "\n\n".join(paras[h:]))):
                    sub = self._map(label, lec, piece, f"{part}.{k}", depth + 1)
                    if sub is None:
                        return None
                    out += sub
                self._store(f"split|{label}|{lec.lecture_id}|{part}", stage="split_on_truncation", model=label,
                            lecture_id=lec.lecture_id, part=part, depth=depth + 1)
                return out
        return None

    def _pack(self, label, parts: List[str]) -> List[List[str]]:
        """Greedy groups under the soft input budget, but EVERY group holds at least 2 parts, so each level strictly
        reduces the number of parts (a group of one part could never make progress)."""
        budget = self.cfg.reduce_input_tokens
        groups, cur, n = [], [], 0
        for p in parts:
            t = self.est_tokens(label, len(p) + 12)
            if cur and n + t > budget:
                groups.append(cur); cur, n = [], 0
            cur.append(p); n += t
        groups.append(cur)
        out, carry = [], []
        for g in groups:
            g = carry + g
            carry = []
            if len(g) == 1:
                carry = g
                continue
            out.append(g)
        if carry:
            if out:
                out[-1].extend(carry)
            else:
                out = [carry]
        return out

    def _reduce_call(self, label, lec, part, body, words):
        """One reduce request. If the answer is cut off by the output ceiling (a model that ignores the word target), ask ONCE
        more for 60% of the words. The retry is a separate, recorded request (part name ends with |shorter) and is covered
        by every budget; it is never repeated."""
        mt = self._cfg(label, "reduce_max_tokens")
        r = self.request("gen_reduce", label, lec.lecture_id, part, P.SYSTEM_GEN, P.reduce_prompt(body, words), mt)
        if r is None and self.last_failure.get(label) in ("truncated", "incomplete", "empty_output") and label not in self.stopped:
            r = self.request("gen_reduce", label, lec.lecture_id, part + "|shorter", P.SYSTEM_GEN,
                             P.reduce_prompt(body, max(100, int(words * 0.6))), mt)
        return r

    def _reduce_tree(self, label, lec, parts: List[str], words: int) -> Optional[str]:
        """Combine partial summaries. Groups are packed under cfg.reduce_input_tokens (same for every model) but never
        smaller than 2 parts, so the number of parts at least halves each level and the loop always terminates.
        A model that ignores the word target (writes long outputs) therefore cannot cause an endless loop."""
        level = 0
        while True:
            groups = self._pack(label, parts)
            if len(parts) == 1 or len(groups) <= 1 or level >= self.cfg.max_reduce_levels:
                body = "\n\n".join(f"[Part {i + 1}]\n{p}" for i, p in enumerate(parts))
                self._levels[(label, lec.lecture_id)] = level
                r = self._reduce_call(label, lec, "reduce" if level == 0 else f"reduce_final_L{level}", body, words)
                return r["text"].strip() if r else None
            new = []
            for g_i, g in enumerate(groups):
                g_words = sum(len(p.split()) for p in g)
                target = min(words * 2, max(150, int(0.4 * g_words)))
                cap = self.specs[label].summary_word_cap
                target = min(target, cap) if cap else target
                body = "\n\n".join(f"[Part {i + 1}]\n{p}" for i, p in enumerate(g))
                r = self._reduce_call(label, lec, f"reduce_L{level}_g{g_i}", body, target)
                if not r:
                    return None
                new.append(r["text"].strip())
            parts, level = new, level + 1

    def generate(self, label: str, lec: Lecture) -> Optional[str]:
        key = f"summary|{label}|{lec.lecture_id}"
        with self.lock:
            if key in self.cache:
                return self.cache[key]["text"]
        chunks = chunk_text(lec.text, self._cfg(label, "chunk_words"))
        words = self.summary_words(lec, label)
        text = None
        if len(chunks) == 1:
            r = self.request("gen_final", label, lec.lecture_id, "final", P.SYSTEM_GEN, P.final_prompt(chunks[0], words),
                             self._cfg(label, "reduce_max_tokens"))
            if r:
                text = r["text"].strip()
            elif self.last_failure.get(label) in ("truncated", "empty_output", "incomplete") and label not in self.stopped:
                chunks = chunk_text(lec.text, max(300, len(chunks[0].split()) // 2))     # fall back to map-reduce on halves
            else:
                return None
        if text is None:
            parts = []
            for i, c in enumerate(chunks):
                m = self._map(label, lec, c, f"map{i}")
                if m is None:
                    return None
                parts += m
            text = self._reduce_tree(label, lec, parts, words)
        if not text or not text.strip():
            return None
        self._store(key, stage="summary", model=label, lecture_id=lec.lecture_id, text=text.strip(), n_chunks=len(chunks),
                    target_words=words, chunk_words=self._cfg(label, "chunk_words"),
                    reduce_group_levels=self._levels.get((label, lec.lecture_id), 0))
        return text.strip()

    def gold_generation(self, label: str, lec: Lecture):
        self._begin(label)
        facts = []
        for i, c in enumerate(chunk_text(lec.text, self._cfg(label, "chunk_words"))):
            r = self.request("gold_generation", label, lec.lecture_id, f"chunk{i}", P.SYSTEM_JUDGE, P.gold_prompt(c),
                             self._cfg(label, "judge_max_tokens"),
                             validate=lambda t, f, c=c: V.v_string_list(t, f, source_text=c, number_recall=0.5),
                             retry_hint="\n\nReturn ONLY a valid JSON array of strings with one item for every fact.")
            if r:
                facts += r["parsed"]
        if self._interrupted(label):
            return
        gt = [f.text for f in lec.facts]
        hit = sum(1 for g in gt if any(_tokens_overlap(g, f) >= 0.7 and _tokens_overlap(f, g) >= 0.7 for f in facts))
        self._store(f"gold_eval|{label}|{lec.lecture_id}", stage="gold_eval", model=label, lecture_id=lec.lecture_id,
                    n_extracted=len(facts), n_true_facts=len(gt), recall_vs_truth=round(hit / max(1, len(gt)), 3))

    # ------------------------------------------------------------------ judging
    def _evidence(self, lec: Lecture, claim: str) -> List[str]:
        sents = sentences(lec.text)
        scored = sorted(sents, key=lambda s: -_tokens_overlap(claim, s))
        return scored[: self.cfg.evidence_per_claim]

    def _faith_call(self, judge, lec, tag, batch, stage="judge_faithfulness") -> Dict[int, str]:
        """batch: [{'i','text'}]. Returns {i: verdict}; verdicts the judge failed to give are simply absent."""
        ev = []
        for c in batch:
            for s in self._evidence(lec, c["text"]):
                if s not in ev:
                    ev.append(s)
        ids = [c["i"] for c in batch]
        r = self.request(stage, judge, lec.lecture_id, tag, P.SYSTEM_JUDGE,
                         P.faithfulness_prompt([(c["i"], c["text"]) for c in batch], "\n".join(ev)),
                         self._cfg(judge, "judge_max_tokens"),
                         validate=lambda t, f, ids=ids: V.v_verdicts(t, f, ids),
                         retry_hint="\n\nReturn ONLY a valid JSON array with one item for EVERY numbered claim.")
        return {int(k): v for k, v in r["parsed"].items()} if r else {}

    def _compl_call(self, judge, lec, tag, facts, summary, stage="judge_completeness") -> Dict[int, bool]:
        ids = [f["i"] for f in facts]
        r = self.request(stage, judge, lec.lecture_id, tag, P.SYSTEM_JUDGE,
                         P.completeness_prompt([(f["i"], f["text"]) for f in facts], summary),
                         self._cfg(judge, "judge_max_tokens"),
                         validate=lambda t, f, ids=ids: V.v_covered(t, f, ids),
                         retry_hint="\n\nReturn ONLY a valid JSON array with one item for EVERY numbered fact.")
        return {int(k): v for k, v in r["parsed"].items()} if r else {}

    def judge_validation(self, judge: str, lectures: List[Lecture]):
        """Run one judge over synthetic items with KNOWN answers (planted errors). Verdicts are stored with the item
        metadata so the analysis needs no other file."""
        for idx in self.cfg.validation_lecture_idx:
            if judge in self.stopped:
                return
            lec = lectures[idx]
            key = f"jv_faith|{judge}|{lec.lecture_id}"
            if key not in self.cache:
                self._begin(judge)
                items = faithfulness_items(lec, self.cfg.validation_per_category)
                verdicts = {}
                for b in range(0, len(items), self.cfg.claim_batch):
                    verdicts.update(self._faith_call(judge, lec, f"jv|b{b}", items[b:b + self.cfg.claim_batch], stage="jv_faithfulness"))
                if self._interrupted(judge):
                    return
                self._store(key, stage="jv_faith", model=judge, lecture_id=lec.lecture_id, items=items,
                            verdicts={str(k): v for k, v in verdicts.items()})
            key = f"jv_compl|{judge}|{lec.lecture_id}"
            if key not in self.cache:
                self._begin(judge)
                case = completeness_case(lec)
                covered = {}
                for b in range(0, len(case["facts"]), self.cfg.gold_facts_per_check):
                    covered.update(self._compl_call(judge, lec, f"jv|b{b}", case["facts"][b:b + self.cfg.gold_facts_per_check],
                                                    case["summary"], stage="jv_completeness"))
                if self._interrupted(judge):
                    return
                self._store(key, stage="jv_compl", model=judge, lecture_id=lec.lecture_id, facts=case["facts"],
                            covered={str(k): v for k, v in covered.items()})

    def claim_set(self, extractor: str, gen_label: str, lec: Lecture, summary: str) -> Optional[List[dict]]:
        """Extract claims from the summary in small batches; every reply is validated; failures are counted by type."""
        key = f"claims|{gen_label}|{lec.lecture_id}"
        with self.lock:
            if key in self.cache:
                return self.cache[key]["claims"]
        self._begin(extractor)
        stats = {"calls": 0, "parsed": 0, "malformed": 0, "incomplete": 0, "split_retries": 0, "unresolved_batches": 0}
        claims: List[str] = []

        def run(batch_text: str, tag: str, depth: int) -> bool:
            before = self._count_outcomes(extractor, "judge_claim_extraction", lec.lecture_id, tag)
            r = self.request("judge_claim_extraction", extractor, lec.lecture_id, tag, P.SYSTEM_JUDGE,
                             P.extract_claims_prompt(batch_text),
                             min(self.cfg.claim_max_tokens, self.specs[extractor].judge_max_tokens or self.cfg.claim_max_tokens),
                             validate=lambda t, f: V.v_string_list(t, f, source_text=batch_text),
                             retry_hint="\n\nReturn ONLY a valid JSON array of strings; include every number.")
            after = self._count_outcomes(extractor, "judge_claim_extraction", lec.lecture_id, tag)
            for k in ("ok", "malformed", "incomplete"):
                stats["parsed" if k == "ok" else k] += after[k] - before[k]
            stats["calls"] += sum(after.values()) - sum(before.values())
            if r:
                claims.extend(r["parsed"])
                return True
            lines = batch_text.splitlines()
            if depth < 1 and len(lines) >= 2 and extractor not in self.stopped:       # split the batch and try each half
                stats["split_retries"] += 1
                h = len(lines) // 2
                ok = run("\n".join(lines[:h]), tag + ".a", depth + 1)
                return run("\n".join(lines[h:]), tag + ".b", depth + 1) and ok
            stats["unresolved_batches"] += 1
            return False

        for b, batch in enumerate(V.split_lines(summary, self.cfg.claim_batch_lines)):
            run(batch, f"gen={gen_label}|b{b}", 0)
        if self._interrupted(extractor):
            return None
        seen, uniq = set(), []
        for c in claims:
            k = re.sub(r"\W+", " ", c.lower()).strip()
            if k not in seen:
                seen.add(k); uniq.append(c)
        if not uniq:
            self._store(f"claims_failed|{gen_label}|{lec.lecture_id}", stage="claims_failed", model=extractor,
                        lecture_id=lec.lecture_id, gen_model=gen_label, stats=stats)
            return None
        out = [{"i": i, "text": c, "kind": "extracted"} for i, c in enumerate(uniq[: self.cfg.max_claims])]
        self._store(key, stage="claims", model=extractor, lecture_id=lec.lecture_id, gen_model=gen_label, claims=out, stats=stats)
        return out

    BUDGET_KINDS = ("stopped", "emergency_stop", "lecture_budget", "model_request_budget", "total_request_budget", "total_token_budget")

    def _interrupted(self, label: str) -> bool:
        """True if the label's last request was refused by a stop or a budget. A multi-call result (claim set, gold facts,
        validation verdicts) that was cut short by this must NOT be stored as if complete: a resume would keep the partial
        result forever and blame the model for verdicts it never had the chance to give."""
        return label in self.stopped or self.last_failure.get(label) in self.BUDGET_KINDS

    def _begin(self, label: str):
        self.last_failure.pop(label, None)                # a fresh multi-call task: forget failures of earlier tasks

    def _count_outcomes(self, label, stage, lecture_id, tag) -> Dict[str, int]:
        c = {"ok": 0, "malformed": 0, "incomplete": 0}
        with self.lock:
            rows = list(self.rows_mem)
        for r in rows:
            if (r["model"] == label and r["stage"] == stage and r["lecture_id"] == lecture_id
                    and (r["part"] == tag or r["part"].startswith(tag + ".")) and r["outcome"] in c):
                c[r["outcome"]] += 1
        return c

    def judge_faithfulness(self, judge, gen_label, lec, claims):
        self._begin(judge)
        verdicts = {}
        for b in range(0, len(claims), self.cfg.claim_batch):
            verdicts.update(self._faith_call(judge, lec, f"gen={gen_label}|b{b}", claims[b:b + self.cfg.claim_batch]))
        if self._interrupted(judge):
            return
        self._store(f"faith|{judge}|{gen_label}|{lec.lecture_id}", stage="faith_verdicts", model=judge, gen_model=gen_label,
                    lecture_id=lec.lecture_id, verdicts={str(k): v for k, v in verdicts.items()}, n_claims=len(claims))

    def judge_completeness(self, judge, gen_label, lec, summary):
        self._begin(judge)
        rng = random.Random(f"gold-{lec.lecture_id}")
        facts = rng.sample(lec.facts, k=min(self.cfg.gold_facts_per_check, len(lec.facts)))
        items = [{"i": i, "text": f.text} for i, f in enumerate(facts)]
        cov = self._compl_call(judge, lec, f"gen={gen_label}", items, summary)
        if self._interrupted(judge):
            return
        self._store(f"compl|{judge}|{gen_label}|{lec.lecture_id}", stage="compl_verdicts", model=judge, gen_model=gen_label,
                    lecture_id=lec.lecture_id, covered={str(k): v for k, v in cov.items()}, fact_ids=[f.fact_id for f in facts])

    def judging(self, lectures: List[Lecture], gen_labels: List[str], only_judge: Optional[str] = None):
        """Operational judging of the generated summaries (cost + agreement only; reliability comes from judge_validation)."""
        for idx in self.cfg.judged_lecture_idx:
            lec = lectures[idx]
            for g in gen_labels:
                s = self.cache.get(f"summary|{g}|{lec.lecture_id}")
                if not s:
                    continue
                judges = [j for j in self.cfg.judges_for.get(self.specs[g].provider, [])
                          if j in self.specs and self.specs[j].family != self.specs[g].family]
                if not judges or (only_judge and only_judge not in judges):
                    continue
                if only_judge is None or only_judge == judges[0]:
                    claims = self.claim_set(judges[0], g, lec, s["text"])
                else:
                    claims = self.cache.get(f"claims|{g}|{lec.lecture_id}", {}).get("claims")
                for j in judges:
                    if only_judge and j != only_judge:
                        continue
                    if claims:
                        self.judge_faithfulness(j, g, lec, claims)
                    self.judge_completeness(j, g, lec, s["text"])

    # ------------------------------------------------------------------ full pilot
    def run(self, lectures: List[Lecture], gen_labels: List[str], gold_label: Optional[str],
            judge_labels: List[str], reasoning_variants: Dict[str, Dict[str, dict]], progress=print):
        self.check_budgets()                                         # refuses before anything is sent
        progress("preflight (lists models; no generation)")
        info = self.preflight()
        for lab in list(self.specs):
            if info.get(lab, {}).get("resolved"):
                self.providers[lab].model_id = info[lab]["resolved"]
            else:
                self._stop(lab, "model_unavailable", f"model id not available: wanted {self.specs[lab].wanted_id}")
        progress("reasoning probe (small)")
        self.reasoning_probe(lectures[0], {l: v for l, v in reasoning_variants.items() if l not in self.stopped})

        def model_task(label):
            try:
                if label in gen_labels:
                    for lec in lectures:
                        if label in self.stopped:
                            progress(f"[{label}] STOPPED ({self.stopped_kind.get(label)}): {self.stopped[label][:110]}")
                            break
                        progress(f"[{label}] generation: {lec.lecture_id}")
                        self.generate(label, lec)
                if label == gold_label and label not in self.stopped:
                    for idx in self.cfg.gold_lecture_idx:
                        progress(f"[{label}] gold generation: {lectures[idx].lecture_id}")
                        self.gold_generation(label, lectures[idx])
                if label in judge_labels and label not in self.stopped:
                    progress(f"[{label}] judge validation")
                    self.judge_validation(label, lectures)
            except Exception as e:                                  # one model's crash must not stop the others
                with self.lock:
                    self.errors.append(f"{label}: {type(e).__name__}: {e}")

        for l, why in self.stopped.items():
            progress(f"[{l}] STOPPED ({self.stopped_kind.get(l)}): {why[:110]}")
        labels = [l for l in self.specs if l not in self.stopped]
        progress("phase 1: models run in parallel (each model sequentially, at its own pace)")
        with ThreadPoolExecutor(max_workers=max(1, len(labels))) as ex:
            list(ex.map(model_task, labels))
        progress("phase 2: operational judging of generated summaries")
        extractors = {self.cfg.judges_for.get(self.specs[g].provider, [None])[0] for g in gen_labels}
        order = [j for j in judge_labels if j in extractors] + [j for j in judge_labels if j not in extractors]

        def judge_task(j):
            try:
                self.judging(lectures, gen_labels, only_judge=j)
            except Exception as e:
                with self.lock:
                    self.errors.append(f"{j}: {type(e).__name__}: {e}")
        # extractors first (they create the claim sets the other judges reuse), then the rest in parallel
        first = [j for j in order if j in extractors]
        rest = [j for j in order if j not in extractors]
        for grp in (first, rest):
            with ThreadPoolExecutor(max_workers=max(1, len(grp))) as ex:
                list(ex.map(judge_task, grp))
        state = {"started": self.started, "finished": _now(), "stopped": self.stopped, "stopped_kind": self.stopped_kind,
                 "tokens_run": self.tokens_run, "requests_run": self.requests_run,
                 "retries_exhausted_not_quota": {l: dict(v) for l, v in self.exhausted.items() if v}, "errors": self.errors,
                 "budget": self.budget_state()}
        (self.out / "run_state.json").write_text(json.dumps(state, indent=1), encoding="utf-8")
        progress("done: " + json.dumps(state["stopped"]) if state["stopped"] else "done")
        return state


Writing pilot_project/src/lecsum_pilot/runner.py


In [9]:
%%writefile pilot_project/src/lecsum_pilot/analyze.py
"""Turn the pilot logs into measured per-model/stage usage, judge-validation results, quota reconciliation and an
UPDATED 100-lecture workload estimate. Nothing here calls an API.
Inputs: requests.jsonl, outputs.jsonl (+ observed limits json, + optional dashboard snapshot json).

Token vocabulary used everywhere (never merged, and a missing figure is NEVER read as zero):
  visible output  - tokens of the returned text, when the provider lets us separate them
  reasoning       - tokens the provider REPORTS as reasoning/thinking
  unaccounted     - total - input - output (- reported thinking): tokens counted but not itemised
  unknown         - calls with no usage figures at all (failed calls): usage cannot be known from the API response"""
from __future__ import annotations

import json
import math
import random
import statistics
from collections import Counter, defaultdict
from pathlib import Path
from typing import Dict, List, Optional

from .synth import COMPL_CATEGORIES, FAITH_CATEGORIES, pilot_lectures


def load_jsonl(p: Path) -> List[dict]:
    return [json.loads(l) for l in Path(p).read_text(encoding="utf-8").splitlines() if l.strip()] if Path(p).exists() else []


def _p95(xs):
    xs = sorted(xs)
    return xs[min(len(xs) - 1, int(0.95 * len(xs)))] if xs else None


def wilson(k: int, n: int, z: float = 1.96):
    """95% Wilson interval for a proportion (honest about how little a small sample says)."""
    if n == 0:
        return None
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return [round(max(0.0, c - h), 3), round(min(1.0, c + h), 3)]


def rate(k: int, n: int) -> Dict:
    return {"count": f"{k}/{n}", "rate": round(k / n, 3) if n else None, "ci95": wilson(k, n)}


def _sum_known(rs, key):
    xs = [r[key] for r in rs if r.get(key) is not None]
    return {"sum": sum(xs), "calls_with_figure": len(xs), "calls_without_figure": len(rs) - len(xs)}


def lecture_words_distribution(n=100, seed=2026):
    """Same synthetic mix as tools/feasibility_estimate.py but in WORDS (independent of any tokenizer)."""
    rng, out = random.Random(seed), []
    for _ in range(n):
        u = rng.random()
        med, sig, lo, hi = ((2500, .5, 600, 9000) if u < .5 else (6000, .55, 1500, 25000) if u < .85 else (14000, .4, 6000, 40000))
        out.append(min(hi, max(lo, rng.lognormvariate(math.log(med), sig))))
    return out


# ------------------------------------------------------------------------------------ usage and outcomes
def per_model_stage(rows: List[dict]) -> Dict:
    g = defaultdict(list)
    for r in rows:
        g[(r["model"], r["stage"])].append(r)
    out = {}
    for (m, st), rs in sorted(g.items()):
        ok = [r for r in rs if r["outcome"] == "ok"]
        out.setdefault(m, {})[st] = {
            "attempts": len(rs), "accepted": len(ok), "retries": sum(1 for r in rs if r["attempt"] > 1),
            "outcomes": dict(Counter(r["outcome"] for r in rs)),
            "failure_subtypes": dict(Counter(r.get("subtype") for r in rs if r["outcome"] not in ("ok", "malformed", "incomplete", "empty_output") and r.get("subtype"))),
            "validation": dict(Counter(r.get("validation") for r in rs if r.get("validation"))),
            "retries_exhausted": dict(Counter(r["exhausted"] for r in rs if r.get("exhausted"))),
            "truncated_finish_length": sum(1 for r in rs if (r.get("finish_reason") or "").lower() in ("length", "max_tokens")),
            "input_tokens": _sum_known(rs, "input_tokens"), "output_tokens_as_reported": _sum_known(rs, "output_tokens"),
            "visible_output_tokens": _sum_known(rs, "visible_output_tokens"),
            "reasoning_tokens_reported": _sum_known([r for r in rs if r.get("usage_known")], "reasoning_tokens"),
            "unaccounted_tokens": _sum_known(rs, "unaccounted_tokens"), "total_tokens": _sum_known(rs, "total_tokens"),
            "calls_with_no_usage_figures": sum(1 for r in rs if not r.get("usage_known")),
            "mean_total_tokens_per_accepted_call": round(statistics.mean([r.get("total_tokens") or 0 for r in ok]), 1) if ok else None,
            "latency_mean_s": round(statistics.mean([r["latency_s"] for r in rs]), 2),
            "latency_p95_s": _p95([r["latency_s"] for r in rs]),
            "scheduler_wait_total_s": round(sum(r.get("sched_wait_s") or 0 for r in rs), 1),
            "param_fallbacks": sorted({f for r in rs for f in r.get("param_fallbacks", [])})[:5]}
    return out


def model_overview(rows: List[dict], outputs: List[dict], run_state: Optional[dict]) -> Dict:
    """One block per model: what happened, kept apart: quota vs overload vs rate limit vs invalid output."""
    res = {}
    summaries = Counter(o["model"] for o in outputs if o["stage"] == "summary")
    for m in sorted({r["model"] for r in rows}):
        rs = [r for r in rows if r["model"] == m]
        gen = [r for r in rs if r["stage"] in ("gen_map", "gen_reduce", "gen_final")]
        res[m] = {
            "attempts": len(rs), "accepted": sum(1 for r in rs if r["outcome"] == "ok"),
            "summaries_produced_of_6": summaries.get(m, 0),
            "summary_words_vs_target": [{"lecture": o["lecture_id"], "target_words": o.get("target_words"), "actual_words": len(o["text"].split())}
                                        for o in outputs if o["stage"] == "summary" and o["model"] == m],
            "rate_limit_429_by_dimension": dict(Counter((r.get("rl_dimension") or "unspecified") for r in rs if str(r.get("subtype", "")).startswith("rate_limit_"))),
            "daily_quota_events": sum(1 for r in rs if r["outcome"] == "daily_quota"),
            "overload_503": sum(1 for r in rs if r.get("subtype") == "overloaded_503"),
            "timeouts_network": sum(1 for r in rs if r.get("subtype") == "timeout_network"),
            "retries_exhausted_not_quota": dict(Counter(r["exhausted"] for r in rs if r.get("exhausted"))),
            "invalid_outputs": {k: sum(1 for r in rs if r["outcome"] == k) for k in ("malformed", "incomplete", "empty_output")},
            "local_budget_refusals": sum(1 for r in rs if r["outcome"] == "local_budget_exceeded"),
            "clamped_output_ceilings": sum(1 for r in rs if r.get("clamped")),
            "generation_split_on_truncation": sum(1 for o in outputs if o["stage"] == "split_on_truncation" and o["model"] == m),
            "generation_attempts": len(gen), "generation_accepted": sum(1 for r in gen if r["outcome"] == "ok"),
            "stopped": (run_state or {}).get("stopped", {}).get(m), "stopped_kind": (run_state or {}).get("stopped_kind", {}).get(m)}
    return res


def length_comparability(outputs: List[dict], ratio_band=(0.7, 1.3), spread_limit=1.25) -> Dict:
    """Do the models' final summaries have comparable LENGTH? Models that ignore the word target (or have a different target,
    e.g. Qwen's capped one) differ in how much they can cover and how much they can get wrong, so quality metrics are
    confounded with length. This reports the facts and sets fair_comparison_claim_allowed; it never hides a model."""
    rows = [o for o in outputs if o.get("stage") == "summary"]
    per_model = {}
    for m in sorted({o["model"] for o in rows}):
        rs = [o for o in rows if o["model"] == m]
        words = [len(o["text"].split()) for o in rs]
        ratios = [w / o["target_words"] for w, o in zip(words, rs) if o.get("target_words")]
        per_model[m] = {"n_summaries": len(rs), "mean_words": round(sum(words) / len(words), 1), "min_words": min(words), "max_words": max(words),
                        "mean_actual_over_target": round(sum(ratios) / len(ratios), 2) if ratios else None,
                        "worst_actual_over_target": round(max(ratios), 2) if ratios else None,
                        "targets_differ_from_other_models": any(o.get("target_words") != rs[0].get("target_words") for o in rs[1:]) or None,
                        "max_reduce_group_levels": max((o.get("reduce_group_levels", 0) for o in rs), default=0)}
    by_lec = {}
    for o in rows:
        by_lec.setdefault(o["lecture_id"], {})[o["model"]] = {"words": len(o["text"].split()), "target_words": o.get("target_words")}
    spreads = {}
    for lec, d in by_lec.items():
        w = [v["words"] for v in d.values() if v["words"]]
        if len(w) >= 2:
            spreads[lec] = round(max(w) / min(w), 2)
    reasons = []
    for m, d in per_model.items():
        r = d["mean_actual_over_target"]
        if r is not None and not (ratio_band[0] <= r <= ratio_band[1]):
            reasons.append(f"{m} writes {r}x its word target on average (outside {ratio_band[0]}-{ratio_band[1]}x)")
    for lec, sp in spreads.items():
        if sp > spread_limit:
            reasons.append(f"on {lec} the longest summary is {sp}x the shortest (limit {spread_limit}x)")
    tg = {m: d["mean_words"] for m, d in per_model.items()}
    targets_equal = len({tuple(sorted({o.get('target_words') for o in rows if o['model'] == m})) for m in per_model}) <= 1
    if not targets_equal:
        reasons.append("models were given different word targets (a documented provider constraint, e.g. Qwen's output ceiling)")
    allowed = bool(rows) and not reasons
    return {"per_model": per_model, "max_over_min_words_by_lecture": spreads, "reasons_length_is_a_confound": reasons,
            "fair_comparison_claim_allowed": allowed,
            "statement": ("Summary lengths are comparable within the stated band; length is still reported next to every quality metric."
                          if allowed else
                          "Summary LENGTHS DIFFER between models, so faithfulness/completeness differences are confounded with length. "
                          "Do not claim a fair or like-for-like comparison from these summaries; report length with every score "
                          "(and, for the real study, control length or compare at matched length).")}


def token_breakdown(rows: List[dict]) -> Dict:
    """Per model: visible output / reported reasoning / unaccounted / unknown, over ALL attempts (wasted ones included)."""
    res = {}
    for m in sorted({r["model"] for r in rows}):
        rs = [r for r in rows if r["model"] == m]
        known = [r for r in rs if r.get("usage_known")]
        stat = Counter(r.get("reasoning_status") for r in known)
        res[m] = {"calls": len(rs), "calls_with_usage": len(known), "calls_without_usage_figures": len(rs) - len(known),
                  "input_tokens": sum(r.get("input_tokens") or 0 for r in known),
                  "output_tokens_as_reported": sum(r.get("output_tokens") or 0 for r in known),
                  "visible_output_tokens": _sum_known(known, "visible_output_tokens"),
                  "reasoning_tokens_reported": _sum_known(known, "reasoning_tokens"),
                  "unaccounted_tokens_positive_only": sum(max(0, r.get("unaccounted_tokens") or 0) for r in known),
                  "total_tokens_as_reported": sum(r.get("total_tokens") or 0 for r in known),
                  "reasoning_status_counts": dict(stat),
                  "note": "reasoning_tokens_reported covers only calls where the provider gave a figure; other calls are NOT zero, they are unknown"}
    return res


def reasoning_probe_table(rows):
    t = defaultdict(dict)
    for r in rows:
        if r["stage"] == "probe_reasoning":
            t[r["model"]][r["variant"]] = {"outcome": r["outcome"], "input": r.get("input_tokens"), "output_as_reported": r.get("output_tokens"),
                                           "visible": r.get("visible_output_tokens"), "reasoning_reported": r.get("reasoning_tokens"),
                                           "reasoning_status": r.get("reasoning_status"), "unaccounted": r.get("unaccounted_tokens"),
                                           "total": r.get("total_tokens"), "finish": r.get("finish_reason"),
                                           "response_chars": r.get("response_chars"), "ceiling": r.get("max_tokens"),
                                           "param_fallbacks": r.get("param_fallbacks", [])}
    return dict(t)


def rate_limit_events(rows: List[dict]) -> List[dict]:
    """Every provider rate-limit / too-large answer with its parsed numbers (limit, used, requested) - lets us see
    what 'requested' means relative to our max_tokens and input size."""
    keep = ("seq", "model", "stage", "status", "subtype", "rl_dimension", "rl_limit", "rl_used", "rl_requested", "rl_retry_in_s",
            "est_input_tokens", "input_tokens", "max_tokens", "reserved_output_tokens", "backoff_s")
    return [{k: r.get(k) for k in keep} for r in rows if str(r.get("subtype", "")).startswith(("rate_limit_", "request_too_large")) or r["outcome"] == "daily_quota"]


def header_vs_log(rows: List[dict]) -> Dict:
    """Does the provider's remaining-tokens header drop by what the usage field reports? (consecutive calls < 30 s apart, same model)"""
    res = {}
    for m in sorted({r["model"] for r in rows}):
        rs = sorted([r for r in rows if r["model"] == m and r["outcome"] == "ok" and str(r["headers"].get("x-ratelimit-remaining-tokens", "")).isdigit()],
                    key=lambda r: r["seq"])
        ratios = []
        for a, b in zip(rs, rs[1:]):
            drop = int(a["headers"]["x-ratelimit-remaining-tokens"]) - int(b["headers"]["x-ratelimit-remaining-tokens"])
            if drop > 0 and b.get("total_tokens"):
                ratios.append(drop / b["total_tokens"])
        if ratios:
            res[m] = {"pairs": len(ratios), "median_header_drop_over_reported_total": round(statistics.median(ratios), 2)}
    return res


# ------------------------------------------------------------------------------------ judge validation
def _kappa(pairs):
    """Cohen's kappa for two binary raters; pairs = [(a, b)]. None when undefined."""
    n = len(pairs)
    if not n:
        return None
    po = sum(a == b for a, b in pairs) / n
    pa, pb = sum(a for a, _ in pairs) / n, sum(b for _, b in pairs) / n
    pe = pa * pb + (1 - pa) * (1 - pb)
    return None if pe == 1 else round((po - pe) / (1 - pe), 3)


def judge_validation_report(outputs: List[dict]) -> Dict:
    faith = defaultdict(dict)       # judge -> {(lecture, i): (item, verdict|None)}
    for o in outputs:
        if o["stage"] == "jv_faith":
            for it in o["items"]:
                faith[o["model"]][(o["lecture_id"], it["i"])] = (it, o["verdicts"].get(str(it["i"])))
    compl = defaultdict(dict)
    for o in outputs:
        if o["stage"] == "jv_compl":
            for f in o["facts"]:
                compl[o["model"]][(o["lecture_id"], f["i"])] = (f, o["covered"].get(str(f["i"])))

    rep = {"faithfulness": {}, "completeness": {}, "pairwise": {"faithfulness": {}, "completeness": {}}}
    for j, d in sorted(faith.items()):
        per_cat = {}
        for cat in FAITH_CATEGORIES:
            items = [(it, v) for it, v in d.values() if it["category"] == cat]
            ans = [(it, v) for it, v in items if v is not None]
            if cat.startswith("supported"):
                per_cat[cat] = {"items": len(items), "answered": len(ans), "judged_supported": rate(sum(v == "supported" for _, v in ans), len(ans))}
            else:
                per_cat[cat] = {"items": len(items), "answered": len(ans),
                                "flagged_not_supported": rate(sum(v != "supported" for _, v in ans), len(ans)),
                                "exact_expected_label": rate(sum(v == it["expected"] for it, v in ans), len(ans))}
        allans = [(it, v) for it, v in d.values() if v is not None]
        correct = sum((it["expected"] == "supported") == (v == "supported") for it, v in allans)
        sup = [(it, v) for it, v in allans if it["expected"] == "supported"]
        err = [(it, v) for it, v in allans if it["expected"] != "supported"]
        rep["faithfulness"][j] = {
            "items": len(d), "answered": len(allans), "no_verdict": len(d) - len(allans), "per_category": per_cat,
            "binary_accuracy_answered": rate(correct, len(allans)),
            "binary_accuracy_missing_counted_wrong": rate(correct, len(d)),
            "false_positive_rate_on_supported": rate(sum(v != "supported" for _, v in sup), len(sup)),
            "miss_rate_on_errors": rate(sum(v == "supported" for _, v in err), len(err))}
    for j, d in sorted(compl.items()):
        per_cat = {}
        for cat in COMPL_CATEGORIES:
            ans = [(f, v) for f, v in d.values() if f["category"] == cat and v is not None]
            per_cat[cat] = {"items": sum(1 for f, _ in d.values() if f["category"] == cat), "answered": len(ans),
                            "correct": rate(sum(v == f["expected"] for f, v in ans), len(ans))}
        allans = [(f, v) for f, v in d.values() if v is not None]
        rep["completeness"][j] = {"items": len(d), "answered": len(allans), "no_answer": len(d) - len(allans), "per_category": per_cat,
                                  "accuracy_answered": rate(sum(v == f["expected"] for f, v in allans), len(allans))}

    def pairs(data, label_of, correct_of, flag_of):
        out = {}
        names = sorted(data)
        for a in range(len(names)):
            for b in range(a + 1, len(names)):
                ka = {k for k, (_, v) in data[names[a]].items() if v is not None}
                kb = {k for k, (_, v) in data[names[b]].items() if v is not None}
                both = sorted(ka & kb)
                if not both:
                    continue
                same = sum(label_of(data[names[a]][k][1]) == label_of(data[names[b]][k][1]) for k in both)
                fl = [(flag_of(data[names[a]][k][1]), flag_of(data[names[b]][k][1])) for k in both]
                wa = [not correct_of(*data[names[a]][k]) for k in both]
                wb = [not correct_of(*data[names[b]][k]) for k in both]
                either = sum(x or y for x, y in zip(wa, wb))
                out[f"{names[a]} vs {names[b]}"] = {
                    "items_answered_by_both": len(both), "identical_label": rate(same, len(both)),
                    "disagreements": len(both) - same, "cohen_kappa_binary": _kappa(fl),
                    "both_wrong_when_either_wrong": rate(sum(x and y for x, y in zip(wa, wb)), either),
                    "reading": "a high 'both wrong' share means the two judges make the SAME mistakes (not independent evidence)"}
        return out
    rep["pairwise"]["faithfulness"] = pairs(faith, lambda v: v, lambda it, v: (it["expected"] == "supported") == (v == "supported"), lambda v: v == "supported")
    rep["pairwise"]["completeness"] = pairs(compl, lambda v: v, lambda f, v: v == f["expected"], lambda v: bool(v))
    n_items = max((len(d) for d in faith.values()), default=0)
    rep["limits"] = (f"{n_items} synthetic faithfulness items per judge; intervals are 95% Wilson. Synthetic planted errors are easier and more "
                     "regular than real summaries. These numbers do NOT establish judge reliability on real lectures, and judges "
                     "from one provider are not independent of each other.")
    return rep


def operational_judging(outputs: List[dict]) -> Dict:
    """Judging of generated summaries: claim-extraction statistics and judge agreement (no planted errors here)."""
    ext = Counter()
    for o in outputs:
        if o["stage"] in ("claims", "claims_failed"):
            for k, v in o["stats"].items():
                ext[k] += v
            ext["claim_sets_ok" if o["stage"] == "claims" else "claim_sets_failed"] += 1
    faith = defaultdict(dict)
    for o in outputs:
        if o["stage"] == "faith_verdicts":
            faith[(o["gen_model"], o["lecture_id"])][o["model"]] = o["verdicts"]
    agree = defaultdict(lambda: [0, 0])
    for k, js in faith.items():
        names = sorted(js)
        for a in range(len(names)):
            for b in range(a + 1, len(names)):
                for i in set(js[names[a]]) & set(js[names[b]]):
                    agree[(names[a], names[b])][1] += 1
                    agree[(names[a], names[b])][0] += js[names[a]][i] == js[names[b]][i]
    compl = defaultdict(dict)
    for o in outputs:
        if o["stage"] == "compl_verdicts":
            compl[(o["gen_model"], o["lecture_id"])][o["model"]] = o["covered"]
    cag = defaultdict(lambda: [0, 0])
    for k, js in compl.items():
        names = sorted(js)
        for a in range(len(names)):
            for b in range(a + 1, len(names)):
                for i in set(js[names[a]]) & set(js[names[b]]):
                    cag[(names[a], names[b])][1] += 1
                    cag[(names[a], names[b])][0] += js[names[a]][i] == js[names[b]][i]
    return {"claim_extraction": dict(ext),
            "faithfulness_agreement": {" vs ".join(k): rate(*v) for k, v in agree.items()},
            "completeness_agreement": {" vs ".join(k): rate(*v) for k, v in cag.items()},
            "note": "agreement is not accuracy; accuracy comes only from judge_validation"}


# ------------------------------------------------------------------------------------ workload and calendar
def workload_update(rows: List[dict], outputs: List[dict], limits: Optional[dict], n_lectures=100) -> Dict:
    lectures = pilot_lectures()
    words_by_id = {l.lecture_id: l.n_words for l in lectures}
    cpw = statistics.mean(len(l.text) / l.n_words for l in lectures)
    gen_stages = ("gen_map", "gen_final", "gen_reduce")
    models = sorted({r["model"] for r in rows if r["stage"] in gen_stages})
    sums = defaultdict(set)
    for o in outputs:
        if o["stage"] == "summary":
            sums[o["model"]].add(o["lecture_id"])
    dist = lecture_words_distribution(n_lectures)
    W = sum(dist)
    tpw, per_word, gen_tokens, gen_reqs = {}, {}, {}, {}
    for m in models:
        gr = [r for r in rows if r["model"] == m and r["stage"] in gen_stages]
        xs = [(r["prompt_chars"], r["input_tokens"]) for r in gr if r["stage"] in ("gen_map", "gen_final") and r["outcome"] == "ok" and r.get("input_tokens")]
        if len(xs) >= 3 and len({x for x, _ in xs}) > 1:
            mx, my = statistics.mean(x for x, _ in xs), statistics.mean(y for _, y in xs)
            b = sum((x - mx) * (y - my) for x, y in xs) / sum((x - mx) ** 2 for x, _ in xs)
            tpw[m] = round(b * cpw, 3)
        else:
            tpw[m] = None
        lecs = sums.get(m, set())
        words = sum(words_by_id.get(l, 0) for l in lecs)
        lec_rows = [r for r in gr if r["lecture_id"] in lecs]
        if words and lec_rows:
            tot = sum(r.get("total_tokens") or 0 for r in lec_rows)
            acc = sum(r.get("total_tokens") or 0 for r in lec_rows if r["outcome"] == "ok")
            per_word[m] = {"tokens_per_source_word_all_attempts": round(tot / words, 3), "tokens_per_source_word_accepted_only": round(acc / words, 3),
                           "wasted_token_share": round(1 - acc / tot, 3) if tot else None,
                           "requests_per_1000_words_all_attempts": round(1000 * len(lec_rows) / words, 3),
                           "lectures_measured": len(lecs), "lectures_attempted": len({r["lecture_id"] for r in gr}),
                           "calls_without_usage_figures": sum(1 for r in lec_rows if not r.get("usage_known"))}
            gen_tokens[m] = round(per_word[m]["tokens_per_source_word_all_attempts"] * W)
            gen_reqs[m] = round(per_word[m]["requests_per_1000_words_all_attempts"] * W / 1000)
        else:
            per_word[m] = None                     # no summary produced: NOT zero cost, simply not measurable
    # judge cost per summary (operational judging only)
    n_sets = Counter(o["model"] for o in outputs if o["stage"] == "claims")
    n_judged = defaultdict(set)
    for o in outputs:
        if o["stage"] in ("faith_verdicts", "compl_verdicts"):
            n_judged[o["model"]].add((o["gen_model"], o["lecture_id"]))
    judge_cost = {}
    for jm in sorted({r["model"] for r in rows if r["stage"].startswith("judge_")}):
        judge_cost[jm] = {}
        for st in ("judge_claim_extraction", "judge_faithfulness", "judge_completeness"):
            rs = [r for r in rows if r["model"] == jm and r["stage"] == st]
            den = n_sets.get(jm, 0) if st == "judge_claim_extraction" else len(n_judged.get(jm, ()))
            if rs and den:
                judge_cost[jm][st] = {"tokens_per_summary": round(sum(r.get("total_tokens") or 0 for r in rs) / den),
                                      "requests_per_summary": round(len(rs) / den, 2), "summaries_measured": den,
                                      "attempts_not_accepted": sum(1 for r in rs if r["outcome"] != "ok")}
    gold = [r for r in rows if r["stage"] == "gold_generation"]
    gold_est = None
    if gold:
        gw = sum(words_by_id.get(l, 0) for l in {r["lecture_id"] for r in gold})
        t = sum(r.get("total_tokens") or 0 for r in gold)
        gold_est = {"model": gold[0]["model"], "tokens_per_source_word": round(t / gw, 3), "estimated_tokens_100_lectures": round(t / gw * W)}
    n_att = len(rows)
    n_bad = sum(1 for r in rows if r["outcome"] != "ok")
    return {"measured_tokens_per_word": tpw, "generation_per_word": per_word,
            "attempt_failure_rate_all_models": round(n_bad / n_att, 3) if n_att else None,
            "scaled_to_lectures": n_lectures, "total_source_words_scaled": round(W),
            "generation_tokens_100_lectures_per_model": gen_tokens, "generation_requests_100_lectures_per_model": gen_reqs,
            "judge_cost_per_summary": judge_cost, "gold_generation": gold_est,
            "not_measured": "QA answering/grading stages were not run in the pilot; extrapolate from completeness-judge cost and label as an assumption",
            "limits_used": limits and {"source": "user console screenshots 2026-11-03"}}


def calendar_days(workload: Dict, limits: Dict, gen_models: List[str], judges: Dict[str, List[str]], n_lectures=100,
                  safety=0.8, minute_limits: Optional[Dict[str, Dict]] = None) -> Dict:
    """Days per model = max(tokens/TPD, requests/RPD) at `safety` of the limit; plus a lower bound in hours from the per-minute
    limits (Qwen's 1,000 output tokens/minute). Models without a measured generation cost are listed, not guessed."""
    lim = {**limits.get("groq_free_plan_per_model", {}), **limits.get("gemini_free_tier_default_project", {})}
    load = defaultdict(lambda: {"tokens": 0.0, "requests": 0.0, "out_tokens": 0.0})
    notes = []
    for g in gen_models:
        gt = workload["generation_tokens_100_lectures_per_model"].get(g)
        if gt is None:
            notes.append(f"{g}: no summary was produced in the pilot, so its generation workload is not estimated")
        else:
            load[g]["tokens"] += gt
            load[g]["requests"] += workload["generation_requests_100_lectures_per_model"].get(g, 0)
        for k, jm in enumerate(judges.get(g, [])):
            for st, v in workload["judge_cost_per_summary"].get(jm, {}).items():
                if st == "judge_claim_extraction" and k > 0:
                    continue
                load[jm]["tokens"] += v["tokens_per_summary"] * n_lectures
                load[jm]["requests"] += v["requests_per_summary"] * n_lectures
    out = {}
    for m, l in load.items():
        key = next((k for k in lim if m.lower().replace("-", " ").replace("_", " ") in k.lower().replace("-", " ").replace("/", " ")), None)
        e = lim.get(key, {})
        days_tok = l["tokens"] / (e["tpd"] * safety) if e.get("tpd") else None
        days_req = l["requests"] / (e["rpd"] * safety) if e.get("rpd") else None
        ml = (minute_limits or {}).get(m, {})
        hours = None
        if ml.get("otpm") or ml.get("itpm") or ml.get("tpm"):
            cands = [l["tokens"] / (ml["tpm"] * safety)] if ml.get("tpm") else []
            if ml.get("otpm"):                                 # assume the measured ~35% output share of total tokens
                cands.append(0.35 * l["tokens"] / (ml["otpm"] * safety))
            hours = round(max(cands) / 60, 1) if cands else None
        out[m] = {"tokens": round(l["tokens"]), "requests": round(l["requests"]), "limit_entry": key,
                  "days_by_daily_tokens": round(days_tok, 1) if days_tok else None,
                  "days_by_daily_requests": round(days_req, 1) if days_req else None,
                  "days": round(max(x for x in (days_tok, days_req, 0) if x is not None), 1),
                  "min_hours_from_per_minute_limits": hours,
                  "unverified": "no daily token cap shown for this provider" if not e.get("tpd") else None}
    return {"per_model": out, "calendar_days_if_models_run_in_parallel": max((v["days"] for v in out.values()), default=None),
            "safety_fraction_of_limits": safety, "notes": notes,
            "excluded": "gold generation (Gemini) and any re-runs after failures are not included; add the failure overhead measured in the pilot"}


# ------------------------------------------------------------------------------------ quota reconciliation
DASHBOARD_TEMPLATE = {
    "_how_to_fill": ("Copy the provider usage numbers by hand BEFORE and AFTER the run (same page, same day). Leave a field null if the "
                     "dashboard does not show it. requests/input_tokens/output_tokens/total_tokens are the day totals shown for that model. "
                     "Note anything else that used the same key or organisation in between."),
    "captured_before_utc": None, "captured_after_utc": None, "other_usage_in_between": None,
    "models": {}}


def write_dashboard_template(path: str, labels: List[str]):
    t = json.loads(json.dumps(DASHBOARD_TEMPLATE))
    blank = {"requests": None, "input_tokens": None, "output_tokens": None, "total_tokens": None}
    t["models"] = {l: {"before": dict(blank), "after": dict(blank), "dashboard_label": None} for l in labels}
    Path(path).write_text(json.dumps(t, indent=1), encoding="utf-8")


def reconcile_with_dashboard(rows: List[dict], dashboard: dict) -> Dict:
    """Compare what OUR logs say we used with what the provider dashboard says changed. Differences are reported, not explained away."""
    res = {}
    for m, d in (dashboard.get("models") or {}).items():
        rs = [r for r in rows if r["model"] == m]
        known = [r for r in rs if r.get("usage_known")]
        delta = {}
        for k in ("requests", "input_tokens", "output_tokens", "total_tokens"):
            b, a = (d.get("before") or {}).get(k), (d.get("after") or {}).get(k)
            delta[k] = (a - b) if isinstance(a, (int, float)) and isinstance(b, (int, float)) else None
        log = {"attempts_all": len(rs), "attempts_with_usage_figures": len(known), "attempts_without_usage_figures": len(rs) - len(known),
               "input_tokens": sum(r.get("input_tokens") or 0 for r in known), "output_tokens_as_reported": sum(r.get("output_tokens") or 0 for r in known),
               "visible_output_tokens": _sum_known(known, "visible_output_tokens"), "reasoning_tokens_reported": _sum_known(known, "reasoning_tokens"),
               "unaccounted_tokens": sum(max(0, r.get("unaccounted_tokens") or 0) for r in known),
               "total_tokens": sum(r.get("total_tokens") or 0 for r in known)}
        notes = []
        if delta["total_tokens"] is not None:
            diff = delta["total_tokens"] - log["total_tokens"]
            log["dashboard_minus_log_total_tokens"] = diff
            tol = 0.05 * max(1, log["total_tokens"])
            if diff > tol:
                notes.append("dashboard counts MORE tokens than the API usage fields we logged: candidates are tokens of calls whose "
                             f"response carried no usage ({log['attempts_without_usage_figures']} such attempts), unreported reasoning, "
                             "or other use of the same key/organisation")
            elif diff < -tol:
                notes.append("dashboard counts FEWER tokens than our logs: the provider may not count some calls (e.g. rejected requests) "
                             "or the dashboard lags")
            else:
                notes.append("dashboard and log agree within 5%")
        if delta["requests"] is not None:
            nd = delta["requests"] - log["attempts_all"]
            log["dashboard_minus_log_requests"] = nd
            if nd < 0:
                notes.append(f"dashboard counts {-nd} fewer requests than we sent (rejected/failed calls may not be counted)")
            elif nd > 0:
                notes.append(f"dashboard counts {nd} more requests than we logged (other usage of the same key, or a counting difference)")
        res[m] = {"dashboard_delta": delta, "log": log, "observations": notes}
    res["_caveat"] = ("Dashboards are rounded/lagging and may aggregate other use of the same key. Treat a mismatch as a question to "
                      "investigate, not as proof of a cause.")
    return res


# ------------------------------------------------------------------------------------ summary
def build_summary(out_dir: str, limits_path: Optional[str] = None, gen_models=None, judges=None,
                  dashboard_path: Optional[str] = None, minute_limits: Optional[Dict] = None) -> Dict:
    out = Path(out_dir)
    rows, outputs = load_jsonl(out / "requests.jsonl"), load_jsonl(out / "outputs.jsonl")
    limits = json.loads(Path(limits_path).read_text()) if limits_path and Path(limits_path).exists() else None
    run_state = json.loads((out / "run_state.json").read_text()) if (out / "run_state.json").exists() else None
    wl = workload_update(rows, outputs, limits)
    summary = {"requests_logged": len(rows), "model_overview": model_overview(rows, outputs, run_state),
               "per_model_stage": per_model_stage(rows), "token_breakdown": token_breakdown(rows),
               "reasoning_probe": reasoning_probe_table(rows), "rate_limit_events": rate_limit_events(rows),
               "header_vs_logged_usage": header_vs_log(rows), "judge_validation": judge_validation_report(outputs),
               "operational_judging": operational_judging(outputs),
               "summary_length_comparability": length_comparability(outputs),
               "gold_generation_eval": [o for o in outputs if o["stage"] == "gold_eval"], "workload_update": wl,
               "run_state": run_state,
               "resolved_models": json.loads((out / "resolved_models.json").read_text()) if (out / "resolved_models.json").exists() else None}
    if limits and gen_models and judges:
        summary["calendar_estimate"] = calendar_days(wl, limits, gen_models, judges, minute_limits=minute_limits)
    if dashboard_path and Path(dashboard_path).exists():
        summary["quota_reconciliation"] = reconcile_with_dashboard(rows, json.loads(Path(dashboard_path).read_text()))
    (out / "pilot_summary.json").write_text(json.dumps(summary, indent=1, ensure_ascii=False), encoding="utf-8")
    return summary


Writing pilot_project/src/lecsum_pilot/analyze.py


In [10]:
%%writefile pilot_project/src/lecsum_pilot/setup.py
"""Pilot wiring (rerun): model list with per-model limits, judge matrix, probe variants, real/mock factories."""
import os
from typing import Dict, List, Optional

from .mock import FakeClock, MockProvider
from .providers import GeminiProvider, GroqProvider
from .runner import FAMILIES, BudgetError, ModelSpec, Pilot, PilotConfig

# Fraction of each free daily limit this run may use (safety). Groq TPD = 200K per model, Gemini Flash-Lite RPD = 500
# (the user's console screenshots, 2026-11-03).
GROQ_TPD, GEMINI_RPD = 200_000, 500
GROQ_RUN_FRACTION, GEMINI_RUN_FRACTION = 0.70, 0.60
GROQ_RUN_REQUEST_CAP = 600

# Per-model protocol settings. Every deviation from the shared default is a documented provider constraint:
#  gpt-oss : reasoning_effort=low; output ceiling raised so reasoning cannot use it all; total request must fit 8,000 TPM.
#  qwen    : Groq splits the per-minute limit (input 7,000 / output 1,000): chunks of 550 words (~850 tokens in; the rerun showed
#            1,300-word chunks need more than 850 output tokens) and output ceilings of 850 tokens, summaries limited to 450 words.
#  gemini  : defaults, higher output ceiling because thinking tokens count against maxOutputTokens.
GROQ_COMMON = dict(rpm=30, tpm=8000)

# ---- Safety budgets for ONE rerun. All counts are HTTP attempts (retries included); they are cumulative over every
# resume that uses the same output folder, so resuming can never spend them twice. NOT provider limits: these are my
# own starting values = about 2x what a complete, healthy MOCK run needs (mock: 423 requests, ~560k estimated tokens,
# at most 23 requests for one model on one lecture). Real behaviour differs; change them on purpose, never by accident.
RECOMMENDED_BUDGET = {
    "total_requests": 800,
    "total_tokens": 900_000,
    "per_model_requests": {"gpt-oss-120b": 200, "gpt-oss-20b": 150, "qwen3.8-27b": 250,
                           "gemini-3.1-flash-lite": 200, "gemini-3.5-flash-lite": 250},
    "per_lecture_requests": {"gen": 40, "gold": 12, "claims_judge": 60, "jv": 30, "probe": 8},   # per (model, lecture, task family)
    "lecture_trips_before_model_stop": 2,
}
MODEL_LABELS = ("gpt-oss-120b", "gpt-oss-20b", "qwen3.8-27b", "gemini-3.1-flash-lite", "gemini-3.5-flash-lite")


def validate_budget(budget) -> dict:
    """Return the budget if every field is a positive whole number, else raise BudgetError (nothing has been sent)."""
    if not isinstance(budget, dict):
        raise BudgetError("No safety budget given. Set BUDGET (e.g. BUDGET = RECOMMENDED_BUDGET, or your own numbers) before "
                          "starting; the run refuses to start without it.")
    good = lambda v: isinstance(v, int) and not isinstance(v, bool) and v > 0
    miss = [k for k in ("total_requests", "total_tokens", "lecture_trips_before_model_stop") if not good(budget.get(k))]
    miss += [f"per_model_requests[{l}]" for l in MODEL_LABELS if not good((budget.get("per_model_requests") or {}).get(l))]
    miss += [f"per_lecture_requests[{f}]" for f in FAMILIES if not good((budget.get("per_lecture_requests") or {}).get(f))]
    if miss:
        raise BudgetError("Safety budget is missing or invalid: " + ", ".join(miss) + ". Nothing was sent.")
    return budget


def make_specs(groq_used_today: Optional[Dict[str, int]] = None, gemini_requests_used_today: Optional[Dict[str, int]] = None,
               per_model_requests: Optional[Dict[str, int]] = None):
    gu, ge = groq_used_today or {}, gemini_requests_used_today or {}
    pm = per_model_requests or {}
    gcap = lambda l: max(0, int(GROQ_TPD * GROQ_RUN_FRACTION) - gu.get(l, 0))
    ecap = lambda l: max(0, int(GEMINI_RPD * GEMINI_RUN_FRACTION) - ge.get(l, 0))
    return [
        ModelSpec("gpt-oss-120b", "groq", "openai/gpt-oss-120b", "openai", max_tokens_per_run=gcap("gpt-oss-120b"),
                  max_requests_per_run=GROQ_RUN_REQUEST_CAP, budget_requests=pm.get("gpt-oss-120b"), extra={"reasoning_effort": "low"},
                  map_max_tokens=3500, reduce_max_tokens=3500, judge_max_tokens=3000, **GROQ_COMMON),
        ModelSpec("gpt-oss-20b", "groq", "openai/gpt-oss-20b", "openai", max_tokens_per_run=gcap("gpt-oss-20b"),
                  max_requests_per_run=GROQ_RUN_REQUEST_CAP, budget_requests=pm.get("gpt-oss-20b"), extra={"reasoning_effort": "low"},
                  map_max_tokens=3500, reduce_max_tokens=3500, judge_max_tokens=3000, **GROQ_COMMON),
        ModelSpec("qwen3.8-27b", "groq", "qwen/qwen3.8-27b", "qwen", max_tokens_per_run=gcap("qwen3.8-27b"),
                  max_requests_per_run=GROQ_RUN_REQUEST_CAP, budget_requests=pm.get("qwen3.8-27b"), extra={"reasoning_effort": "none"},
                  itpm=7000, otpm=1000, chunk_words=550, map_max_tokens=850, reduce_max_tokens=850,
                  judge_max_tokens=850, summary_word_cap=450, **GROQ_COMMON),
        ModelSpec("gemini-3.1-flash-lite", "gemini", "gemini-3.1-flash-lite", "google", rpm=15,
                  max_requests_per_run=ecap("gemini-3.1-flash-lite"), budget_requests=pm.get("gemini-3.1-flash-lite"), max_attempts_overload=6, overload_circuit=3),
        ModelSpec("gemini-3.5-flash-lite", "gemini", "gemini-3.5-flash-lite", "google", rpm=15,
                  max_requests_per_run=ecap("gemini-3.5-flash-lite"), budget_requests=pm.get("gemini-3.5-flash-lite"), max_attempts_overload=6, overload_circuit=3),
    ]


SPECS = make_specs()
GEN_LABELS = [s.label for s in SPECS]                     # all five are tested as generators (Gemini 3.5 = baseline)
JUDGE_LABELS = list(GEN_LABELS)                           # all five are validated as judges on planted errors
# Operational judging: judge family differs from the generator family; the FIRST judge also extracts the claims.
JUDGES_FOR = {"groq": ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite"],
              "gemini": ["gpt-oss-120b", "qwen3.8-27b"]}
GOLD_LABEL = "gemini-3.5-flash-lite"
REASONING_VARIANTS = {
    "gpt-oss-120b": {"effort_low": {"reasoning_effort": "low"}, "effort_medium": {"reasoning_effort": "medium"}},
    "gpt-oss-20b": {"effort_low": {"reasoning_effort": "low"}, "effort_medium": {"reasoning_effort": "medium"}},
    "qwen3.8-27b": {"effort_none": {"reasoning_effort": "none"}, "effort_default": {"reasoning_effort": "default"}},
}


def make_config(out_dir: str, budget: Optional[dict] = None) -> PilotConfig:
    cfg = PilotConfig(out_dir=out_dir, judges_for=JUDGES_FOR, validation_per_category=8, judged_lecture_idx=(1, 3))
    if budget:
        cfg.max_total_requests, cfg.max_total_tokens = budget["total_requests"], budget["total_tokens"]
        cfg.per_lecture_requests = dict(budget["per_lecture_requests"])
        cfg.lecture_trips_before_model_stop = budget["lecture_trips_before_model_stop"]
    return cfg


def real_pilot(out_dir: str, groq_key: str, gemini_key: str, groq_used_today=None, gemini_requests_used_today=None, budget=None, sleep=None) -> Pilot:
    if groq_used_today is None or gemini_requests_used_today is None:
        raise ValueError("Enter today's usage first (Groq: tokens per model from the Usage page; Gemini: requests per model from "
                         "AI Studio). Without it the safety caps assume nothing was used today, which is how the second run "
                         "hit the daily limits.")
    missing = [l for l in ("gpt-oss-120b", "gpt-oss-20b", "qwen3.8-27b") if groq_used_today.get(l) is None] + \
              [l for l in ("gemini-3.1-flash-lite", "gemini-3.5-flash-lite") if gemini_requests_used_today.get(l) is None]
    if missing:
        raise ValueError(f"Usage today is missing for: {missing}. Use 0 only if the console really shows 0.")
    validate_budget(budget)                                  # no budget, no pilot (raised before any provider object exists)
    specs = make_specs(groq_used_today, gemini_requests_used_today, budget["per_model_requests"])
    provs = {s.label: (GroqProvider(s.label, s.wanted_id, groq_key) if s.provider == "groq"
                       else GeminiProvider(s.label, s.wanted_id, gemini_key)) for s in specs}
    pilot = Pilot(make_config(out_dir, budget), specs, provs, **({"sleep": sleep} if sleep else {}))
    pilot.check_budgets()
    return pilot


def mock_pilot(out_dir: str, sleep=None, specs=None, budget=None, **mock_kwargs) -> Pilot:
    """Fake providers + a virtual clock. mock_kwargs: {label: MockProvider options}. By default the Groq-like mocks
    enforce the same per-minute limits as the real specs so the scheduler is really exercised."""
    budget = budget or RECOMMENDED_BUDGET
    specs = specs or make_specs(per_model_requests=budget["per_model_requests"])
    clock = FakeClock()
    provs = {}
    for s in specs:
        kw = dict(mock_kwargs.get(s.label, {}))
        if "limits" not in kw and s.provider == "groq":
            kw["limits"] = {"rpm": s.rpm, "tpm": s.tpm, **({"itpm": s.itpm} if s.itpm else {}), **({"otpm": s.otpm} if s.otpm else {})}
        provs[s.label] = MockProvider(s.label, s.wanted_id, **kw)
    pilot = Pilot(make_config(out_dir, budget), specs, provs, sleep=sleep or clock.sleep, clock=clock.now)
    for p in provs.values():
        p.time_fn = pilot.now
    return pilot


Writing pilot_project/src/lecsum_pilot/setup.py


In [11]:
%%writefile pilot_project/tools/observed_free_tier_limits_2026-11-03.json
{
 "note": "Read from the user's own console screenshots on 2026-11-03 (Cairo). Account-specific, free tier; providers can change them. Models showing 0/0 are not usable.",
 "groq_free_plan_per_model": {
  "openai/gpt-oss-120b": {"rpm": 30, "rpd": 1000, "tpm": 8000, "tpd": 200000},
  "openai/gpt-oss-20b": {"rpm": 30, "rpd": 1000, "tpm": 8000, "tpd": 200000},
  "qwen/qwen3.8-27b": {"rpm": 30, "rpd": 1000, "tpm": 8000, "tpd": 200000},
  "allam-2-7b": {"rpm": 30, "rpd": 7000, "tpm": 6000, "tpd": 500000}
 },
 "gemini_free_tier_default_project": {
  "Gemini 3.1 Flash Lite": {"rpm": 15, "tpm": 250000, "rpd": 500},
  "Gemini 3.5 Flash Lite": {"rpm": 15, "tpm": 250000, "rpd": 500},
  "Gemini 2.5 Flash Lite": {"rpm": 10, "tpm": 250000, "rpd": 20},
  "Gemini 2.5 Flash": {"rpm": 5, "tpm": 250000, "rpd": 20},
  "Gemini 3 Flash": {"rpm": 5, "tpm": 250000, "rpd": 20},
  "Gemini 3.6 Flash": {"rpm": 5, "tpm": 250000, "rpd": 20},
  "Gemini 3.8 Flash": {"rpm": 5, "tpm": 250000, "rpd": 20},
  "Gemma 4 26B": {"rpm": 30, "tpm": 16000, "rpd": 14400},
  "Gemma 4 31B": {"rpm": 30, "tpm": 16000, "rpd": 14400},
  "unusable_0_0": ["Gemini 2 Flash", "Gemini 2 Flash Lite", "Gemini 2.5 Pro", "Gemini 3.1 Pro", "Gemini 4 Argon", "Gemini Omni 1.1 Flash"]
 }
}


Writing pilot_project/tools/observed_free_tier_limits_2026-11-03.json


In [12]:
import sys, os, json, getpass, collections
from datetime import datetime, timezone
sys.path.insert(0, "pilot_project/src")
from lecsum_pilot.setup import *
from lecsum_pilot.synth import pilot_lectures
from lecsum_pilot.analyze import build_summary, load_jsonl, write_dashboard_template
lectures = pilot_lectures()
print("synthetic lectures:", [(l.lecture_id, l.n_words) for l in lectures])

synthetic lectures: [('syn-databases-1500-7', 1125), ('syn-operating_systems-2500-7', 1892), ('syn-microeconomics-4000-7', 2966), ('syn-organic_chemistry-6000-7', 4454), ('syn-signals_systems-9000-7', 6680), ('syn-machine_learning-14000-7', 10377)]


### B - keys (hidden boxes; never printed, never saved)
Paste a key only into the box that appears. If you ever pasted a key anywhere else, delete it in the provider's console and make a new one.

In [13]:
groq_key = getpass.getpass("Groq API key (hidden): ")
gemini_key = getpass.getpass("Gemini API key (hidden): ")
print("keys entered:", bool(groq_key), bool(gemini_key))
OUT_DIR = "/kaggle/working/pilot_out_v2"            # on Kaggle only /kaggle/working is saved
os.makedirs(OUT_DIR, exist_ok=True)

Groq API key (hidden):  ········
Gemini API key (hidden):  ········


keys entered: True True


### C - today's usage, safety budget, and preflight (this cell makes NO generation calls)
**You must replace every `None`** with the number shown TODAY: Groq console -> Usage (tokens per model) and AI Studio -> Rate limit / Usage (requests per model). Use `0` only if the page really shows 0. The cell refuses to continue while any `None` is left.

`BUDGET = RECOMMENDED_BUDGET` = 800 requests, 900,000 tokens, per-model and per-lecture caps (my starting values, not provider limits). Delete the line to be refused.

In [14]:
GROQ_USED_TODAY = {"gpt-oss-120b": 0, "gpt-oss-20b": 0, "qwen3.8-27b": 0}
GEMINI_REQUESTS_USED_TODAY = {"gemini-3.1-flash-lite": 0, "gemini-3.5-flash-lite": 0}
BUDGET = RECOMMENDED_BUDGET

BEFORE = {m: {"requests": None, "input_tokens": None, "output_tokens": None, "total_tokens": None} for m in GEN_LABELS}   # optional dashboard numbers
captured_before = datetime.now(timezone.utc).isoformat(timespec="seconds")

pilot = real_pilot(OUT_DIR, groq_key, gemini_key, groq_used_today=GROQ_USED_TODAY,
                   gemini_requests_used_today=GEMINI_REQUESTS_USED_TODAY, budget=BUDGET)
del groq_key, gemini_key
info = pilot.preflight()
for k, v in info.items():
    print(k, "->", v)
print()
for s in pilot.specs.values():
    print(f"{s.label:24s} token cap {s.max_tokens_per_run}  request cap today {s.max_requests_per_run}  rerun budget {s.budget_requests}")
print("whole-rerun budget:", pilot.cfg.max_total_requests, "requests,", pilot.cfg.max_total_tokens, "tokens")
print("\nIf every model above shows a resolved id, run cell D. If any shows resolved: None, stop and send me this output.")

gpt-oss-120b -> {'wanted': 'openai/gpt-oss-120b', 'resolved': 'openai/gpt-oss-120b', 'alternatives': []}
gpt-oss-20b -> {'wanted': 'openai/gpt-oss-20b', 'resolved': 'openai/gpt-oss-20b', 'alternatives': []}
qwen3.8-27b -> {'wanted': 'qwen/qwen3.8-27b', 'resolved': 'qwen/qwen3.8-27b', 'alternatives': []}
gemini-3.1-flash-lite -> {'wanted': 'gemini-3.1-flash-lite', 'resolved': 'gemini-3.1-flash-lite', 'alternatives': []}
gemini-3.5-flash-lite -> {'wanted': 'gemini-3.5-flash-lite', 'resolved': 'gemini-3.5-flash-lite', 'alternatives': []}

gpt-oss-120b             token cap 140000  request cap today 600  rerun budget 200
gpt-oss-20b              token cap 140000  request cap today 600  rerun budget 150
qwen3.8-27b              token cap 140000  request cap today 600  rerun budget 250
gemini-3.1-flash-lite    token cap None  request cap today 300  rerun budget 200
gemini-3.5-flash-lite    token cap None  request cap today 300  rerun budget 250
whole-rerun budget: 800 requests, 900000 tokens

### D - RUN (real API calls, synthetic text only)
Expect 1-2 hours. Keep this tab open. If it stops, run this cell again to resume. Do **not** enable billing if a model stops.

In [15]:
state = pilot.run(lectures, GEN_LABELS, GOLD_LABEL, JUDGE_LABELS, REASONING_VARIANTS)
print("\nstopped models:", json.dumps(state["stopped"], indent=1))
print("stop kinds     :", state["stopped_kind"])
print("retries exhausted (NOT quota):", state["retries_exhausted_not_quota"])
print("errors         :", state["errors"])
print("budget state   :", json.dumps(state["budget"], indent=1))
captured_after = datetime.now(timezone.utc).isoformat(timespec="seconds")

preflight (lists models; no generation)
reasoning probe (small)
phase 1: models run in parallel (each model sequentially, at its own pace)
[gpt-oss-120b] generation: syn-databases-1500-7
[gpt-oss-20b] generation: syn-databases-1500-7
[qwen3.8-27b] generation: syn-databases-1500-7
[gemini-3.1-flash-lite] generation: syn-databases-1500-7
[gemini-3.5-flash-lite] generation: syn-databases-1500-7
[gpt-oss-20b] generation: syn-operating_systems-2500-7
[gpt-oss-120b] generation: syn-operating_systems-2500-7
[gemini-3.1-flash-lite] generation: syn-operating_systems-2500-7
[gemini-3.5-flash-lite] generation: syn-operating_systems-2500-7
[gemini-3.1-flash-lite] generation: syn-microeconomics-4000-7
[gemini-3.5-flash-lite] generation: syn-microeconomics-4000-7
[gpt-oss-20b] generation: syn-microeconomics-4000-7
[gpt-oss-120b] generation: syn-microeconomics-4000-7
[gemini-3.5-flash-lite] generation: syn-organic_chemistry-6000-7
[qwen3.8-27b] generation: syn-operating_systems-2500-7
[gpt-oss-20b] g

### E - analyse and zip
Optional: type the dashboard numbers AFTER the run in `AFTER` (wait ~5 minutes), then run. Send me the zip and screenshots of both usage pages.

In [17]:
# Patch for cell E (analysis only, no API calls): ignore requests the budget refused, they were never sent
import lecsum_pilot.analyze as A
_orig_load = A.load_jsonl
def _load(path):
    rows = _orig_load(path)
    if str(path).endswith("requests.jsonl"):
        rows = [r for r in rows if r.get("outcome") != "budget_refused"]
    return rows
A.load_jsonl = _load

In [18]:
import shutil
AFTER = {m: {"requests": None, "input_tokens": None, "output_tokens": None, "total_tokens": None} for m in GEN_LABELS}
dash = {"captured_before_utc": captured_before, "captured_after_utc": captured_after, "other_usage_in_between": "none",
        "models": {m: {"before": BEFORE[m], "after": AFTER[m]} for m in GEN_LABELS}}
json.dump(dash, open(OUT_DIR + "/dashboard_usage.json", "w"), indent=1)
minute_limits = {s.label: {"tpm": s.tpm, "itpm": s.itpm, "otpm": s.otpm} for s in pilot.specs.values()}
summary = build_summary(OUT_DIR, "pilot_project/tools/observed_free_tier_limits_2026-11-03.json", GEN_LABELS, JUDGES_FOR,
                        dashboard_path=OUT_DIR + "/dashboard_usage.json", minute_limits=minute_limits)
for k in ["model_overview", "token_breakdown", "reasoning_probe", "quota_reconciliation"]:
    print("==", k); print(json.dumps(summary.get(k), indent=1)[:3500])
print("== summary length comparability"); print(json.dumps(summary["summary_length_comparability"], indent=1)[:3500])
print("== judge validation (per judge)"); print(json.dumps({j: {"binary_accuracy": v["binary_accuracy_answered"], "no_verdict": v["no_verdict"]} for j, v in summary["judge_validation"]["faithfulness"].items()}, indent=1))
print("== calendar estimate"); print(json.dumps(summary.get("calendar_estimate"), indent=1)[:3000])
print("\nzip:", shutil.make_archive("/kaggle/working/pilot_results_v2", "zip", OUT_DIR), "(find it under Output on the right)")

== model_overview
{
 "gemini-3.1-flash-lite": {
  "attempts": 59,
  "accepted": 50,
  "summaries_produced_of_6": 6,
  "summary_words_vs_target": [
   {
    "lecture": "syn-databases-1500-7",
    "target_words": 150,
    "actual_words": 150
   },
   {
    "lecture": "syn-operating_systems-2500-7",
    "target_words": 150,
    "actual_words": 135
   },
   {
    "lecture": "syn-microeconomics-4000-7",
    "target_words": 177,
    "actual_words": 271
   },
   {
    "lecture": "syn-organic_chemistry-6000-7",
    "target_words": 267,
    "actual_words": 897
   },
   {
    "lecture": "syn-signals_systems-9000-7",
    "target_words": 400,
    "actual_words": 609
   },
   {
    "lecture": "syn-machine_learning-14000-7",
    "target_words": 622,
    "actual_words": 591
   }
  ],
  "rate_limit_429_by_dimension": {},
  "daily_quota_events": 0,
  "overload_503": 7,
  "timeouts_network": 2,
  "retries_exhausted_not_quota": {},
  "invalid_outputs": {
   "malformed": 0,
   "incomplete": 0,
   "empty_o